# VERITAS — Notebook 4 of 4 — Full VERITAS

**Assigned to:** Member D  
**This notebook trains only:** `full_veritas`

The other three variants are **not** trained here. After every member finishes, merge results with `VERITAS_05_compare.ipynb`.

## Shared protocol (must be identical on all four notebooks)

Do **not** change these if you want a valid comparison:

- `run_mode` (all `smoke` together, or all `full` together)
- `seed` (keep `42`)
- `split_protocol` / `split_pool`
- `image_size`
- `classification_threshold`
- `face_margin`
- `max_train_images` / `max_val_images` / `max_test_images`

`experiments_to_run` is already locked to `["full_veritas"]`.

Dataset: attach `nathanielescuro/veritas-openforensics-compiled`. Follow `KAGGLE_RUN_GUIDE.md`.


## 1. Environment


In [ ]:
import os
import sys
import platform
import subprocess
import warnings

warnings.filterwarnings("ignore", category=UserWarning)

def _pip_install(pkgs):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *pkgs])

# Install only if missing. Do not upgrade preinstalled Kaggle stacks.
_required = {
    "torch": "torch",
    "torchvision": "torchvision",
    "cv2": "opencv-python-headless",
    "PIL": "Pillow",
    "sklearn": "scikit-learn",
    "pandas": "pandas",
    "matplotlib": "matplotlib",
    "scipy": "scipy",
    "tqdm": "tqdm",
    "pyarrow": "pyarrow",
}
for import_name, pip_name in _required.items():
    try:
        __import__(import_name if import_name != "cv2" else "cv2")
    except Exception:
        print(f"Installing missing package: {pip_name}")
        _pip_install([pip_name])

try:
    import mediapipe  # noqa: F401
except Exception:
    print("Installing mediapipe (current Tasks API, not deprecated solutions)")
    try:
        _pip_install(["mediapipe"])
    except Exception as e:
        print("mediapipe install failed; detector will use a non-MediaPipe fallback:", e)

import torch
import torchvision
import cv2
import numpy as np
import pandas as pd
from PIL import Image

print("Python:", sys.version.split()[0], platform.platform())
print("PyTorch:", torch.__version__)
print("torchvision:", torchvision.__version__)
print("OpenCV:", cv2.__version__)
print("NumPy:", np.__version__)
print("Pandas:", pd.__version__)
print("PIL:", Image.__version__ if hasattr(Image, "__version__") else getattr(Image, "PILLOW_VERSION", "?"))
try:
    import mediapipe as mp
    print("MediaPipe:", mp.__version__)
except Exception as e:
    mp = None
    print("MediaPipe: unavailable", e)

print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    props = torch.cuda.get_device_properties(0)
    print(f"VRAM: {props.total_memory / (1024**3):.2f} GB")
else:
    print("GPU: CPU fallback")


## 2. Imports


In [ ]:
import gc
import json
import math
import random
import shutil
import time
import hashlib
import traceback
from collections import defaultdict, OrderedDict
from dataclasses import dataclass, asdict, field
from pathlib import Path
from typing import Any, Dict, List, Optional, Sequence, Tuple

import matplotlib.pyplot as plt
from matplotlib.patches import Polygon as MplPolygon, Rectangle
from tqdm.auto import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from torchvision.models import efficientnet_b7, EfficientNet_B7_Weights

from sklearn.metrics import (
    accuracy_score,
    f1_score,
    precision_score,
    recall_score,
    confusion_matrix,
)
from scipy import stats

try:
    from mediapipe.tasks.python import vision as mp_vision
    from mediapipe.tasks.python.core.base_options import BaseOptions as MpBaseOptions
    MP_TASKS_AVAILABLE = True
except Exception as e:
    mp_vision = None
    MpBaseOptions = None
    MP_TASKS_AVAILABLE = False
    print("MediaPipe Tasks API not importable:", e)

%matplotlib inline
plt.rcParams["figure.dpi"] = 120


## 3. Dataset discovery


In [ ]:
def _is_kaggle() -> bool:
    return Path("/kaggle/input").exists() or Path("/kaggle/working").exists()


def discover_paths() -> Dict[str, Path]:
    # Inspect actual filesystem. Never hard-code a dataset folder name.
    search_roots: List[Path] = []
    cwd = Path.cwd()
    if Path("/kaggle/input").exists():
        search_roots.append(Path("/kaggle/input"))
    search_roots.extend(
        [
            cwd / "dataset",
            cwd / "dataset" / "compiled",
            cwd,
        ]
    )

    found_parquet = None
    found_poly = None
    found_images = None
    found_missing = None
    found_extra = None
    walked = []

    for root in search_roots:
        if not root.exists():
            continue
        for dirpath, dirnames, filenames in os.walk(root):
            walked.append(dirpath)
            names = set(filenames)
            if found_parquet is None and "compiled_index.parquet" in names:
                found_parquet = Path(dirpath) / "compiled_index.parquet"
            if found_poly is None and "compiled_poly.json" in names:
                found_poly = Path(dirpath) / "compiled_poly.json"
            if found_poly is None and "poly.json" in names:
                found_poly = Path(dirpath) / "poly.json"
            if found_missing is None and "missing_images.csv" in names:
                found_missing = Path(dirpath) / "missing_images.csv"
            if found_extra is None and "extra_files.csv" in names:
                found_extra = Path(dirpath) / "extra_files.csv"
            if found_images is None:
                if "Images" in dirnames:
                    found_images = Path(dirpath) / "Images"
                elif any(fn.lower().endswith((".jpg", ".jpeg", ".png")) for fn in filenames[:20]) and Path(dirpath).name.lower() in {
                    "images",
                    "train",
                    "val",
                    "test-dev",
                    "test-challenge",
                }:
                    found_images = Path(dirpath)
            # Do not enumerate 100k JPEGs during discovery.
            if Path(dirpath).name == "Images":
                dirnames[:] = []
            if found_parquet and found_images and found_poly:
                break
        if found_parquet and found_images:
            break

    in_kaggle = Path("/kaggle/input").exists()
    working = Path("/kaggle/working") if in_kaggle else (cwd / "kaggle_working")
    working.mkdir(parents=True, exist_ok=True)

    print("Walked roots:", [str(r) for r in search_roots if r.exists()])
    print("parquet:", found_parquet)
    print("poly.json:", found_poly)
    print("Images:", found_images)
    print("working:", working)
    if found_images is None or (found_parquet is None and found_poly is None):
        raise FileNotFoundError(
            "Could not locate compiled OpenForensics files. "
            "Attach dataset nathanielescuro/veritas-openforensics-compiled on Kaggle."
        )
    return {
        "parquet": found_parquet,
        "poly": found_poly,
        "images": found_images,
        "missing_csv": found_missing,
        "extra_csv": found_extra,
        "working": working,
        "dataset_root": found_images.parent if found_images is not None else None,
    }


PATHS = discover_paths()
for sub in ["checkpoints", "predictions", "masks", "debug", "logs"]:
    (PATHS["working"] / sub).mkdir(parents=True, exist_ok=True)
print("Artifact dirs ready under", PATHS["working"])


## 4. Dataset structure


In [ ]:
def sample_image_files(image_dir: Path, n: int = 8) -> List[Path]:
    out = []
    for p in image_dir.iterdir():
        if p.is_file() and p.suffix.lower() in {".jpg", ".jpeg", ".png"}:
            out.append(p)
            if len(out) >= n:
                break
    return out


print("Image directory listing (sample):")
for p in sample_image_files(PATHS["images"]):
    print(" ", p.name, p.stat().st_size, "bytes")

subdirs = [p.name for p in PATHS["images"].iterdir() if p.is_dir()]
print("Image subdirectories:", subdirs[:20], "(empty means flat Images/ layout)")


## 5. Central configuration


In [ ]:
CONFIG: Dict[str, Any] = {
    # "smoke" = tiny subset so the notebook finishes; "full" = research protocol
    "run_mode": "smoke",
    "seed": 42,
    "backbone": "efficientnet_b7",
    "use_pretrained": True,
    "image_size": 600,  # EfficientNet-B7 native; smoke mode may downscale
    "face_margin": 0.10,
    "batch_size": 4,
    "num_workers": 2 if os.name != "nt" else 0,
    "epochs": 8,
    "learning_rate": 1e-4,
    "weight_decay": 1e-4,
    "classification_threshold": 0.5,
    "lambda_cls": 1.0,
    "lambda_seg": 1.0,
    "dice_weight": 1.0,  # L_seg = BCE + dice_weight * Dice
    "use_amp": True,
    "gradient_accumulation": 1,
    "match_iou_threshold": 0.3,
    "face_score_threshold": 0.5,
    "split_protocol": "70_15_15",  # SOP default; "official_openforensics" also supported
    "split_pool": "train_val_testdev",  # do not mix Test-Challenge into 70/15/15 unless "all"
    "max_train_images": None,
    "max_val_images": None,
    "max_test_images": None,
    "skip_training": False,
    "resume": True,
    "experiments_to_run": ["full_veritas"],  # locked: this notebook trains only this variant
    "loss_configs": [
        {"name": "1_1", "lambda_cls": 1.0, "lambda_seg": 1.0},
        {"name": "2_1", "lambda_cls": 2.0, "lambda_seg": 1.0},
        {"name": "1_2", "lambda_cls": 1.0, "lambda_seg": 2.0},
    ],
    "run_loss_sweeps": False,  # set True in full mode to sweep λ on full_veritas
    "latency_warmup": 5,
    "latency_repeats": 20,
}

EXPERIMENTS = {
    "baseline_effb7": {
        "multi_stream": False,
        "segmentation": False,
        "backbone": "efficientnet_b7",
    },
    "multistream_no_seg": {
        "multi_stream": True,
        "segmentation": False,
        "backbone": "efficientnet_b7",
    },
    "seg_no_multistream": {
        "multi_stream": False,
        "segmentation": True,
        "backbone": "efficientnet_b7",
    },
    "full_veritas": {
        "multi_stream": True,
        "segmentation": True,
        "backbone": "efficientnet_b7",
    },
}

if CONFIG["run_mode"] == "smoke":
    CONFIG.update(
        {
            "image_size": 320,
            "batch_size": 2,
            "epochs": 1,
            "max_train_images": 24,
            "max_val_images": 8,
            "max_test_images": 8,
            "latency_warmup": 1,
            "latency_repeats": 3,
            "run_loss_sweeps": False,
            "use_pretrained": True,
        }
    )
    print("SMOKE MODE: reduced image_size/epochs/subset. Set run_mode='full' for the thesis protocol.")
    print("SOP native EfficientNet-B7 size is 600; smoke uses", CONFIG["image_size"], "to stay runnable.")

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", DEVICE)
print("CONFIG:")
for k, v in CONFIG.items():
    print(f"  {k}: {v}")


In [ ]:
# Shared-protocol fingerprint. All four members must print the SAME hash.
import hashlib, json as _json
_protocol = {k: CONFIG[k] for k in [
    "run_mode", "seed", "image_size", "face_margin", "classification_threshold",
    "split_protocol", "split_pool", "max_train_images", "max_val_images", "max_test_images",
]}
_blob = _json.dumps(_protocol, sort_keys=True, default=str)
PROTOCOL_HASH = hashlib.sha256(_blob.encode()).hexdigest()[:16]
print("THIS NOTEBOOK TRAINS:", CONFIG["experiments_to_run"])
print("PROTOCOL_HASH:", PROTOCOL_HASH)
print(_json.dumps(_protocol, indent=2))
(PATHS["working"] / "logs" / "protocol.json").write_text(
    _json.dumps({"hash": PROTOCOL_HASH, "protocol": _protocol, "experiment": CONFIG["experiments_to_run"]}, indent=2),
    encoding="utf-8",
)


## 6. Reproducibility


In [ ]:
def set_seed(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


set_seed(CONFIG["seed"])
print("Seed locked to", CONFIG["seed"])


## 7. Annotation inspection


In [ ]:
CATEGORY_NAME = {0: "real", 1: "fake"}


def parse_segmentation(seg: Any) -> List[List[Tuple[float, float]]]:
    # Parse COCO-style segmentation from parquet string or native JSON.
    if seg is None or (isinstance(seg, float) and math.isnan(seg)):
        return []
    if isinstance(seg, str):
        seg = json.loads(seg)
    polygons: List[List[Tuple[float, float]]] = []
    if not isinstance(seg, list):
        return polygons
    for poly in seg:
        if poly is None or len(poly) == 0:
            continue
        if isinstance(poly[0], (list, tuple)):
            pts = [(float(p[0]), float(p[1])) for p in poly if len(p) >= 2]
        else:
            vals = [float(x) for x in poly]
            pts = [(vals[i], vals[i + 1]) for i in range(0, len(vals) - 1, 2)]
        if len(pts) >= 3:
            polygons.append(pts)
    return polygons


def load_annotation_index(paths: Dict[str, Path]) -> pd.DataFrame:
    if paths.get("parquet") is not None and Path(paths["parquet"]).exists():
        df = pd.read_parquet(paths["parquet"])
        print("Loaded parquet index:", df.shape)
    elif paths.get("poly") is not None:
        print("Loading poly.json (large); prefer parquet when available...")
        with open(paths["poly"], "r", encoding="utf-8") as f:
            raw = json.load(f)
        images = {im["id"]: im for im in raw.get("images", [])}
        rows = []
        for ann in raw.get("annotations", []):
            im = images.get(ann["image_id"], {})
            bbox = ann.get("bbox", [0, 0, 0, 0])
            rows.append(
                {
                    "ann_id": ann.get("id"),
                    "image_id": ann.get("image_id"),
                    "split": im.get("split", "unknown"),
                    "file_name": im.get("file_name"),
                    "category_id": ann.get("category_id"),
                    "bbox_x": bbox[0],
                    "bbox_y": bbox[1],
                    "bbox_w": bbox[2],
                    "bbox_h": bbox[3],
                    "area": ann.get("area", 0),
                    "segmentation": json.dumps(ann.get("segmentation", [])),
                    "orig_ann_id": ann.get("id"),
                    "width": im.get("width"),
                    "height": im.get("height"),
                }
            )
        df = pd.DataFrame(rows)
        print("Loaded poly.json:", df.shape)
        print("categories:", raw.get("categories"))
    else:
        raise FileNotFoundError("No parquet or poly.json")
    print("columns:", list(df.columns))
    print(df.head(2).to_string())
    print("\ncategory_id counts:\n", df["category_id"].value_counts())
    print("\nsplit counts (faces):\n", df["split"].value_counts())
    print("\nunique images:", df["image_id"].nunique())
    print("faces/image mean:", df.groupby("image_id").size().mean())
    mixed = (df.groupby("image_id")["category_id"].nunique() > 1).sum()
    print("mixed real/fake images:", int(mixed))
    return df


ANN_DF = load_annotation_index(PATHS)

# Verify a representative schema row
_schema_row = ANN_DF.iloc[0]
print("\nSchema example:")
print("  image_id:", _schema_row["image_id"])
print("  file_name:", _schema_row["file_name"])
print("  category_id:", int(_schema_row["category_id"]), "=>", CATEGORY_NAME.get(int(_schema_row["category_id"])))
print("  bbox xywh:", [_schema_row["bbox_x"], _schema_row["bbox_y"], _schema_row["bbox_w"], _schema_row["bbox_h"]])
_polys = parse_segmentation(_schema_row["segmentation"])
print("  n polygons:", len(_polys), "first n pts:", 0 if not _polys else len(_polys[0]))
print("  coordinate convention: x=horizontal, y=vertical, absolute pixels, origin top-left")


## 8. Face-level data model, geometry, and coordinate transforms


In [ ]:
@dataclass
class FaceSample:
    image_id: int
    face_id: int
    bbox_xyxy: Tuple[float, float, float, float]
    label: int  # 0 real, 1 fake
    file_name: str
    image_path: str
    polygon: List[List[Tuple[float, float]]]
    ann_id: Optional[int] = None
    split_hint: str = ""
    match_status: str = "ground_truth"
    detector_score: float = 1.0

    @property
    def is_fake(self) -> bool:
        return int(self.label) == 1


def xywh_to_xyxy(x: float, y: float, w: float, h: float) -> Tuple[float, float, float, float]:
    return float(x), float(y), float(x + w), float(y + h)


def clip_bbox(x1, y1, x2, y2, width, height):
    x1 = min(max(x1, 0.0), width - 1.0)
    y1 = min(max(y1, 0.0), height - 1.0)
    x2 = min(max(x2, x1 + 1.0), width)
    y2 = min(max(y2, y1 + 1.0), height)
    return x1, y1, x2, y2


def expand_bbox(x1, y1, x2, y2, width, height, margin: float):
    bw, bh = x2 - x1, y2 - y1
    x1e = x1 - bw * margin
    y1e = y1 - bh * margin
    x2e = x2 + bw * margin
    y2e = y2 + bh * margin
    return clip_bbox(x1e, y1e, x2e, y2e, width, height)


def bbox_iou(a, b) -> float:
    ax1, ay1, ax2, ay2 = a
    bx1, by1, bx2, by2 = b
    ix1, iy1 = max(ax1, bx1), max(ay1, by1)
    ix2, iy2 = min(ax2, bx2), min(ay2, by2)
    iw, ih = max(0.0, ix2 - ix1), max(0.0, iy2 - iy1)
    inter = iw * ih
    area_a = max(0.0, ax2 - ax1) * max(0.0, ay2 - ay1)
    area_b = max(0.0, bx2 - bx1) * max(0.0, by2 - by1)
    union = area_a + area_b - inter
    return inter / union if union > 0 else 0.0


def stable_face_id_order(boxes: Sequence[Tuple[float, float, float, float]]) -> List[int]:
    """Spatial order: top-to-bottom, then left-to-right. Never detector list order."""
    keyed = []
    for i, (x1, y1, x2, y2) in enumerate(boxes):
        keyed.append(((y1 + y2) * 0.5, (x1 + x2) * 0.5, i))
    keyed.sort()
    return [i for _, _, i in keyed]


class GeometryTransform:
    """One geometric mapping applied to image, bbox, polygon, and mask.

    Pipeline:
        original image pixels
            -> expand/clip face crop (offset x1,y1)
            -> letterbox resize into size x size (scale + pad)
            -> model tensor
    """

    def __init__(self, size: int, face_margin: float = 0.1):
        self.size = int(size)
        self.face_margin = float(face_margin)

    def crop_window(self, bbox_xyxy, image_w, image_h):
        x1, y1, x2, y2 = bbox_xyxy
        return expand_bbox(x1, y1, x2, y2, image_w, image_h, self.face_margin)

    def letterbox_params(self, crop_w: float, crop_h: float) -> Dict[str, float]:
        scale = min(self.size / max(crop_h, 1e-6), self.size / max(crop_w, 1e-6))
        nw = crop_w * scale
        nh = crop_h * scale
        left = (self.size - nw) / 2.0
        top = (self.size - nh) / 2.0
        return {"scale": scale, "left": left, "top": top, "nw": nw, "nh": nh, "crop_w": crop_w, "crop_h": crop_h}

    def apply_image(self, image_rgb: np.ndarray, bbox_xyxy) -> Tuple[np.ndarray, Dict[str, float]]:
        h, w = image_rgb.shape[:2]
        x1, y1, x2, y2 = self.crop_window(bbox_xyxy, w, h)
        x1i, y1i, x2i, y2i = map(int, [round(x1), round(y1), round(x2), round(y2)])
        x1i, y1i, x2i, y2i = clip_bbox(x1i, y1i, x2i, y2i, w, h)
        x1i, y1i, x2i, y2i = int(x1i), int(y1i), int(round(x2i)), int(round(y2i))
        crop = image_rgb[y1i:y2i, x1i:x2i]
        if crop.size == 0:
            crop = np.zeros((self.size, self.size, 3), dtype=np.uint8)
            meta = {"x1": x1, "y1": y1, "x2": x2, "y2": y2, **self.letterbox_params(1, 1)}
            return crop, meta
        ch, cw = crop.shape[:2]
        params = self.letterbox_params(cw, ch)
        nw, nh = int(round(params["nw"])), int(round(params["nh"]))
        nw, nh = max(nw, 1), max(nh, 1)
        resized = cv2.resize(crop, (nw, nh), interpolation=cv2.INTER_LINEAR)
        canvas = np.zeros((self.size, self.size, 3), dtype=np.uint8)
        left = int(round(params["left"]))
        top = int(round(params["top"]))
        left = min(max(left, 0), self.size - nw)
        top = min(max(top, 0), self.size - nh)
        canvas[top:top + nh, left:left + nw] = resized
        meta = {
            "x1": float(x1i), "y1": float(y1i), "x2": float(x2i), "y2": float(y2i),
            "scale": params["scale"], "left": float(left), "top": float(top),
            "nw": float(nw), "nh": float(nh), "crop_w": float(cw), "crop_h": float(ch),
            "orig_w": float(w), "orig_h": float(h),
        }
        return canvas, meta

    def polygon_to_model(self, polygons, meta) -> List[List[Tuple[float, float]]]:
        out = []
        for poly in polygons:
            pts = []
            for x, y in poly:
                xc = x - meta["x1"]
                yc = y - meta["y1"]
                xm = xc * meta["scale"] + meta["left"]
                ym = yc * meta["scale"] + meta["top"]
                pts.append((xm, ym))
            out.append(pts)
        return out

    def rasterize(self, polygons_model, fake: bool) -> np.ndarray:
        mask = np.zeros((self.size, self.size), dtype=np.uint8)
        if not fake:
            return mask
        for poly in polygons_model:
            if len(poly) < 3:
                continue
            pts = np.array(poly, dtype=np.int32).reshape(-1, 1, 2)
            cv2.fillPoly(mask, [pts], 1)
        return mask

    def mask_to_original(self, mask_hw: np.ndarray, meta, orig_h: int, orig_w: int) -> np.ndarray:
        """Project a model-space mask back onto the original image."""
        full = np.zeros((orig_h, orig_w), dtype=np.float32)
        left, top = int(round(meta["left"])), int(round(meta["top"]))
        nw, nh = int(round(meta["nw"])), int(round(meta["nh"]))
        x1, y1 = int(round(meta["x1"])), int(round(meta["y1"]))
        x2, y2 = int(round(meta["x2"])), int(round(meta["y2"]))
        crop_w, crop_h = max(x2 - x1, 1), max(y2 - y1, 1)
        region = mask_hw[top:top + nh, left:left + nw]
        if region.size == 0:
            return full
        restored = cv2.resize(region.astype(np.float32), (crop_w, crop_h), interpolation=cv2.INTER_LINEAR)
        y2e = min(y1 + crop_h, orig_h)
        x2e = min(x1 + crop_w, orig_w)
        full[y1:y2e, x1:x2e] = restored[: y2e - y1, : x2e - x1]
        return full


GEO = GeometryTransform(CONFIG["image_size"], CONFIG["face_margin"])
print("GeometryTransform size", GEO.size, "margin", GEO.face_margin)
print("Coordinate convention: x=horizontal, y=vertical, mask shape=[H,W]")


## 9. Face detector abstraction (MediaPipe Tasks, no deprecated solutions API)


In [ ]:
class FaceDetector:
    """Backend-agnostic detector. Output contract is a list of dicts."""

    name = "base"

    def detect(self, image_rgb: np.ndarray) -> List[Dict[str, Any]]:
        raise NotImplementedError


class MediaPipeTasksFaceDetector(FaceDetector):
    """Current MediaPipe Tasks Face Detector (not mediapipe.solutions)."""

    name = "mediapipe_tasks_blaze_face"

    MODEL_URLS = [
        "https://storage.googleapis.com/mediapipe-models/face_detector/blaze_face_short_range/float16/latest/blaze_face_short_range.tflite",
        "https://storage.googleapis.com/mediapipe-models/face_detector/blaze_face_short_range/float16/1/blaze_face_short_range.tflite",
    ]

    def __init__(self, model_path: Path, min_confidence: float = 0.5):
        if not MP_TASKS_AVAILABLE:
            raise RuntimeError("MediaPipe Tasks API is not available")
        self.model_path = Path(model_path)
        self.min_confidence = min_confidence
        options = mp_vision.FaceDetectorOptions(
            base_options=MpBaseOptions(model_asset_path=str(self.model_path)),
            min_detection_confidence=float(min_confidence),
        )
        self._detector = mp_vision.FaceDetector.create_from_options(options)

    def detect(self, image_rgb: np.ndarray) -> List[Dict[str, Any]]:
        import mediapipe as mp_mod
        if image_rgb.ndim != 3:
            return []
        rgb = np.ascontiguousarray(image_rgb)
        mp_image = mp_mod.Image(image_format=mp_mod.ImageFormat.SRGB, data=rgb)
        result = self._detector.detect(mp_image)
        faces = []
        detections = result.detections or []
        for i, det in enumerate(detections):
            bb = det.bounding_box
            x1, y1 = float(bb.origin_x), float(bb.origin_y)
            x2, y2 = x1 + float(bb.width), y1 + float(bb.height)
            score = 0.0
            if det.categories:
                score = float(det.categories[0].score or 0.0)
            faces.append({"face_id": i, "bbox": [x1, y1, x2, y2], "confidence": score})
        order = stable_face_id_order([f["bbox"] for f in faces]) if faces else []
        restab = []
        for new_id, old in enumerate(order):
            item = faces[old]
            item["face_id"] = new_id
            restab.append(item)
        return restab


class OpenCVHaarFaceDetector(FaceDetector):
    """Last-resort fallback. Documented as weaker than MediaPipe Tasks."""

    name = "opencv_haar"

    def __init__(self, min_confidence: float = 0.5):
        xml = cv2.data.haarcascades + "haarcascade_frontalface_default.xml"
        self.cascade = cv2.CascadeClassifier(xml)
        self.min_confidence = min_confidence

    def detect(self, image_rgb: np.ndarray) -> List[Dict[str, Any]]:
        gray = cv2.cvtColor(image_rgb, cv2.COLOR_RGB2GRAY)
        boxes = self.cascade.detectMultiScale(gray, scaleFactor=1.1, minNeighbors=5, minSize=(24, 24))
        faces = []
        for i, (x, y, w, h) in enumerate(boxes):
            faces.append({"face_id": i, "bbox": [float(x), float(y), float(x + w), float(y + h)], "confidence": 1.0})
        order = stable_face_id_order([f["bbox"] for f in faces]) if faces else []
        return [{**faces[old], "face_id": i} for i, old in enumerate(order)]


class GroundTruthFaceDetector(FaceDetector):
    """Returns annotation boxes. Used when we evaluate classification independent of detection."""

    name = "ground_truth"

    def __init__(self, records_by_image: Dict[int, List[FaceSample]]):
        self.records_by_image = records_by_image

    def detect(self, image_rgb: np.ndarray, image_id: Optional[int] = None) -> List[Dict[str, Any]]:
        if image_id is None:
            return []
        recs = self.records_by_image.get(int(image_id), [])
        return [
            {"face_id": r.face_id, "bbox": list(r.bbox_xyxy), "confidence": 1.0}
            for r in recs
        ]


def download_file(url: str, dest: Path) -> bool:
    dest = Path(dest)
    if dest.exists() and dest.stat().st_size > 1000:
        return True
    dest.parent.mkdir(parents=True, exist_ok=True)
    try:
        import urllib.request
        print("Downloading", url)
        urllib.request.urlretrieve(url, dest)
        return dest.exists() and dest.stat().st_size > 1000
    except Exception as e:
        print("Download failed:", e)
        return False


def build_face_detector() -> FaceDetector:
    model_path = PATHS["working"] / "models" / "blaze_face_short_range.tflite"
    if MP_TASKS_AVAILABLE:
        ok = False
        for url in MediaPipeTasksFaceDetector.MODEL_URLS:
            if download_file(url, model_path):
                ok = True
                break
        if ok:
            try:
                det = MediaPipeTasksFaceDetector(model_path, CONFIG["face_score_threshold"])
                print("Using MediaPipe Tasks FaceDetector:", det.name)
                return det
            except Exception as e:
                print("MediaPipe Tasks init failed:", e)
    print("Falling back to OpenCV Haar cascade. Detection quality will be lower.")
    return OpenCVHaarFaceDetector(CONFIG["face_score_threshold"])


FACE_DETECTOR = build_face_detector()
print("Active detector:", FACE_DETECTOR.name)


## 10. Face / annotation matching


In [ ]:
def match_faces_to_annotations(
    det_boxes: List[Tuple[float, float, float, float]],
    ann_boxes: List[Tuple[float, float, float, float]],
    iou_threshold: float,
) -> Dict[str, Any]:
    """Hungarian-style greedy IoU matching. Detector order is never trusted.

    Policy for unmatched annotation: log it and do not fabricate a detector face.
    Policy for unmatched detection: keep it unlabeled (label=None) for inference;
    drop it from supervised training construction.
    Ambiguous: two dets with IoU to same ann both above threshold after assignment
    of the winner — the loser is unmatched_detector.
    """
    n_d, n_a = len(det_boxes), len(ann_boxes)
    if n_d == 0 and n_a == 0:
        return {"pairs": [], "unmatched_det": [], "unmatched_ann": [], "ambiguous": []}
    iou_mat = np.zeros((n_d, n_a), dtype=np.float32)
    for i, db in enumerate(det_boxes):
        for j, ab in enumerate(ann_boxes):
            iou_mat[i, j] = bbox_iou(db, ab)

    pairs = []
    used_d, used_a = set(), set()
    flat = []
    for i in range(n_d):
        for j in range(n_a):
            flat.append((float(iou_mat[i, j]), i, j))
    flat.sort(reverse=True)
    ambiguous = []
    for iou, i, j in flat:
        if iou < iou_threshold:
            continue
        if i in used_d or j in used_a:
            if i not in used_d and j in used_a and iou >= iou_threshold:
                ambiguous.append({"det_index": i, "ann_index": j, "iou": iou, "reason": "ann_already_taken"})
            continue
        used_d.add(i)
        used_a.add(j)
        pairs.append({"det_index": i, "ann_index": j, "iou": iou})

    unmatched_det = [i for i in range(n_d) if i not in used_d]
    unmatched_ann = [j for j in range(n_a) if j not in used_a]
    return {
        "pairs": pairs,
        "unmatched_det": unmatched_det,
        "unmatched_ann": unmatched_ann,
        "ambiguous": ambiguous,
        "iou_matrix": iou_mat,
    }


# Quick self-check: swapped detector order must still match
_ann = [(10, 10, 50, 50), (200, 20, 260, 90)]
_det = [(198, 18, 262, 92), (12, 11, 49, 51)]  # reversed order
_m = match_faces_to_annotations(_det, _ann, 0.3)
assert _m["pairs"][0]["det_index"] != _m["pairs"][1]["det_index"]
print("Matching self-check:", _m["pairs"])


## 11–12. Polygon → face-crop mask


In [ ]:
def polygon_to_mask(polygon, face_bbox, original_image_size, output_size, is_fake=True, face_margin=None) -> np.ndarray:
    """Rasterize a polygon defined in original image coordinates onto the model grid."""
    geo = GeometryTransform(output_size, CONFIG["face_margin"] if face_margin is None else face_margin)
    orig_h, orig_w = original_image_size
    dummy = np.zeros((orig_h, orig_w, 3), dtype=np.uint8)
    _, meta = geo.apply_image(dummy, face_bbox)
    polys = polygon if polygon and isinstance(polygon[0], (list, tuple)) and polygon and isinstance(polygon[0][0], (list, tuple)) else [polygon] if polygon else []
    # normalize to List[List[Tuple]]
    norm = []
    for poly in (polygon or []):
        if not poly:
            continue
        if isinstance(poly[0], (list, tuple)):
            norm.append([(float(p[0]), float(p[1])) for p in poly])
        else:
            vals = list(poly)
            norm.append([(vals[k], vals[k + 1]) for k in range(0, len(vals) - 1, 2)])
    model_polys = geo.polygon_to_model(norm, meta)
    return geo.rasterize(model_polys, fake=bool(is_fake))


print("polygon_to_mask ready; real faces always receive a zero mask")


## 13. Image path resolution and face-level dataset


In [ ]:
def resolve_image_path(file_name: str, image_dir: Path) -> Optional[Path]:
    raw = str(file_name).replace("\\", "/")
    candidates = [
        image_dir / Path(raw).name,
        image_dir.parent / raw,
        image_dir / raw,
        Path(raw),
    ]
    # extra_files.csv used split subfolders for a handful of extras; try those too
    for split in ("Train", "Val", "Test-Dev", "Test-Challenge"):
        candidates.append(image_dir / split / Path(raw).name)
    for c in candidates:
        if c.exists():
            return c
    return None


def build_missing_set(paths: Dict[str, Path]) -> set:
    s = set()
    if paths.get("missing_csv") and Path(paths["missing_csv"]).exists():
        m = pd.read_csv(paths["missing_csv"])
        if "file_name" in m.columns:
            s |= set(m["file_name"].map(lambda x: Path(str(x)).name))
        if "orig_image_id" in m.columns:
            s |= set(m["orig_image_id"].astype(str))
    return s


MISSING_NAMES = build_missing_set(PATHS)
print("missing_images.csv names:", len(MISSING_NAMES))


def records_from_annotations(df: pd.DataFrame, image_ids: Sequence[int]) -> List[FaceSample]:
    """Supervised samples come from ground-truth boxes, not from the detector.

    Detector order is irrelevant here. face_id is a stable spatial index per image.
    """
    sub = df[df["image_id"].isin(set(map(int, image_ids)))].copy()
    samples: List[FaceSample] = []
    skipped_missing = 0
    for image_id, g in sub.groupby("image_id"):
        rows = list(g.itertuples(index=False))
        boxes = [xywh_to_xyxy(r.bbox_x, r.bbox_y, r.bbox_w, r.bbox_h) for r in rows]
        order = stable_face_id_order(boxes)
        file_name = str(rows[0].file_name)
        path = resolve_image_path(file_name, PATHS["images"])
        if path is None or Path(path).name in MISSING_NAMES:
            skipped_missing += 1
            continue
        for face_id, idx in enumerate(order):
            r = rows[idx]
            samples.append(
                FaceSample(
                    image_id=int(image_id),
                    face_id=int(face_id),
                    bbox_xyxy=boxes[idx],
                    label=int(r.category_id),
                    file_name=file_name,
                    image_path=str(path),
                    polygon=parse_segmentation(r.segmentation),
                    ann_id=int(getattr(r, "ann_id", -1)),
                    split_hint=str(getattr(r, "split", "")),
                    match_status="ground_truth",
                )
            )
    print(f"Built {len(samples)} face samples from {len(set(s.image_id for s in samples))} images; skipped missing {skipped_missing}")
    return samples


class FaceCropDataset(Dataset):
    """Face-level dataset. One item = one face, never a whole-image label."""

    def __init__(self, samples: List[FaceSample], geo: GeometryTransform, augment: bool = False):
        self.samples = samples
        self.geo = geo
        self.augment = augment
        self.imagenet_mean = np.array([0.485, 0.456, 0.406], dtype=np.float32)
        self.imagenet_std = np.array([0.229, 0.224, 0.225], dtype=np.float32)

    def __len__(self):
        return len(self.samples)

    def _augment(self, img: np.ndarray, mask: np.ndarray) -> Tuple[np.ndarray, np.ndarray]:
        if random.random() < 0.5:
            img = np.ascontiguousarray(img[:, ::-1])
            mask = np.ascontiguousarray(mask[:, ::-1])
        if random.random() < 0.5:
            alpha = 1.0 + random.uniform(-0.15, 0.15)
            beta = random.uniform(-12, 12)
            img = np.clip(img.astype(np.float32) * alpha + beta, 0, 255).astype(np.uint8)
        return img, mask

    def __getitem__(self, idx: int) -> Dict[str, Any]:
        s = self.samples[idx]
        bgr = cv2.imread(s.image_path, cv2.IMREAD_COLOR)
        if bgr is None:
            img = np.zeros((self.geo.size, self.geo.size, 3), dtype=np.uint8)
            mask = np.zeros((self.geo.size, self.geo.size), dtype=np.uint8)
            meta = {}
        else:
            rgb = cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)
            img, meta = self.geo.apply_image(rgb, s.bbox_xyxy)
            model_polys = self.geo.polygon_to_model(s.polygon, meta)
            mask = self.geo.rasterize(model_polys, fake=s.is_fake)
        if self.augment:
            img, mask = self._augment(img, mask)
        x = img.astype(np.float32) / 255.0
        x = (x - self.imagenet_mean) / self.imagenet_std
        x = np.transpose(x, (2, 0, 1))
        return {
            "image": torch.from_numpy(x.copy()),
            "mask": torch.from_numpy(mask.astype(np.float32)).unsqueeze(0),
            "label": torch.tensor([float(s.label)], dtype=torch.float32),
            "image_id": s.image_id,
            "face_id": s.face_id,
            "ann_id": -1 if s.ann_id is None else int(s.ann_id),
            "bbox": torch.tensor(s.bbox_xyxy, dtype=torch.float32),
            "path": s.image_path,
        }


def collate_faces(batch):
    out = {
        "image": torch.stack([b["image"] for b in batch], 0),
        "mask": torch.stack([b["mask"] for b in batch], 0),
        "label": torch.stack([b["label"] for b in batch], 0),
        "image_id": [b["image_id"] for b in batch],
        "face_id": [b["face_id"] for b in batch],
        "ann_id": [b["ann_id"] for b in batch],
        "bbox": torch.stack([b["bbox"] for b in batch], 0),
        "path": [b["path"] for b in batch],
    }
    return out


## 14. Visualization helpers


In [ ]:
def load_rgb(path: str) -> np.ndarray:
    bgr = cv2.imread(path, cv2.IMREAD_COLOR)
    if bgr is None:
        raise FileNotFoundError(path)
    return cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)


def draw_full_image(image_rgb, faces: List[Dict[str, Any]], title: str = "", save_path: Optional[Path] = None):
    vis = image_rgb.copy()
    overlay = vis.copy()
    h, w = vis.shape[:2]
    for f in faces:
        x1, y1, x2, y2 = [int(round(v)) for v in f["bbox"]]
        label = str(f.get("label", "")).upper()
        prob = f.get("fake_probability", None)
        is_fake = str(label).lower() in {"fake", "1"} or (prob is not None and prob >= CONFIG["classification_threshold"])
        color = (220, 40, 40) if is_fake else (40, 180, 70)
        cv2.rectangle(vis, (x1, y1), (x2, y2), color, 2)
        text = f"Face {f.get('face_id', '?')} — {label}"
        if prob is not None:
            text += f" — {float(prob):.2f}"
        cv2.putText(vis, text, (x1, max(0, y1 - 8)), cv2.FONT_HERSHEY_SIMPLEX, 0.5, color, 2, cv2.LINE_AA)
        mask = f.get("mask")
        if mask is not None and is_fake:
            if mask.shape[:2] != (h, w):
                continue
            binm = (mask > 0.5).astype(np.uint8)
            overlay[binm == 1] = (0.45 * overlay[binm == 1] + 0.55 * np.array(color)).astype(np.uint8)
    vis = cv2.addWeighted(overlay, 0.45, vis, 0.55, 0)
    fig, ax = plt.subplots(1, 1, figsize=(8, 8))
    ax.imshow(vis)
    ax.set_title(title)
    ax.axis("off")
    if save_path is not None:
        fig.savefig(save_path, bbox_inches="tight")
    plt.show()
    plt.close(fig)
    return vis


print("Visualization helpers ready")


## 15. Dataset split (image-level, no face leakage)


In [ ]:
def existing_image_ids(df: pd.DataFrame) -> pd.DataFrame:
    # Fast path: treat missing_images.csv as the missing set. Resolve paths later per sample.
    uniq = df.drop_duplicates("image_id")[["image_id", "file_name", "split"]].copy()
    uniq["exists"] = ~uniq["file_name"].map(lambda x: Path(str(x)).name in MISSING_NAMES)
    print("images after missing-csv filter:", int(uniq["exists"].sum()), "/", len(uniq))
    return uniq


IMAGE_TABLE = existing_image_ids(ANN_DF)


def split_image_ids(image_table: pd.DataFrame) -> Dict[str, List[int]]:
    protocol = CONFIG["split_protocol"]
    pool = CONFIG["split_pool"]
    table = image_table[image_table["exists"]].copy()
    if pool == "train_val_testdev":
        table = table[table["split"].isin(["Train", "Val", "Test-Dev"])]
    elif pool == "train_only":
        table = table[table["split"] == "Train"]
    elif pool == "all":
        pass
    else:
        raise ValueError(pool)

    rng = np.random.RandomState(CONFIG["seed"])
    if protocol == "official_openforensics":
        splits = {
            "train": sorted(table[table["split"] == "Train"]["image_id"].tolist()),
            "val": sorted(table[table["split"] == "Val"]["image_id"].tolist()),
            "test": sorted(table[table["split"] == "Test-Dev"]["image_id"].tolist()),
        }
        print("Using official OpenForensics Train / Val / Test-Dev")
    elif protocol == "70_15_15":
        ids = table["image_id"].to_numpy().copy()
        rng.shuffle(ids)
        n = len(ids)
        n_train = int(0.70 * n)
        n_val = int(0.15 * n)
        splits = {
            "train": sorted(ids[:n_train].tolist()),
            "val": sorted(ids[n_train:n_train + n_val].tolist()),
            "test": sorted(ids[n_train + n_val:].tolist()),
        }
        print("Using SOP image-level 70/15/15 split on pool", pool, "n=", n)
    else:
        raise ValueError(protocol)

    def cap(key, max_n):
        if max_n is None:
            return splits[key]
        rng2 = np.random.RandomState(CONFIG["seed"] + {"train": 1, "val": 2, "test": 3}[key])
        arr = np.array(splits[key], copy=True)
        rng2.shuffle(arr)
        return sorted(arr[:max_n].tolist())

    splits = {
        "train": cap("train", CONFIG["max_train_images"]),
        "val": cap("val", CONFIG["max_val_images"]),
        "test": cap("test", CONFIG["max_test_images"]),
    }
    assert set(splits["train"]).isdisjoint(splits["val"])
    assert set(splits["train"]).isdisjoint(splits["test"])
    assert set(splits["val"]).isdisjoint(splits["test"])
    for k, v in splits.items():
        print(f"  {k}: {len(v)} images")
    split_path = PATHS["working"] / "logs" / "image_splits.json"
    with open(split_path, "w", encoding="utf-8") as f:
        json.dump({k: list(map(int, v)) for k, v in splits.items()}, f)
    print("Saved", split_path)
    return splits


SPLITS = split_image_ids(IMAGE_TABLE)
TRAIN_SAMPLES = records_from_annotations(ANN_DF, SPLITS["train"])
VAL_SAMPLES = records_from_annotations(ANN_DF, SPLITS["val"])
TEST_SAMPLES = records_from_annotations(ANN_DF, SPLITS["test"])
print("face counts train/val/test:", len(TRAIN_SAMPLES), len(VAL_SAMPLES), len(TEST_SAMPLES))
print("train label balance:", sum(s.label for s in TRAIN_SAMPLES), "fake /", len(TRAIN_SAMPLES))


## 16. Model definitions — configurable VERITAS


In [ ]:
IMAGENET_MEAN = torch.tensor([0.485, 0.456, 0.406]).view(1, 3, 1, 1)
IMAGENET_STD = torch.tensor([0.229, 0.224, 0.225]).view(1, 3, 1, 1)


def _srm_kernels() -> torch.Tensor:
    """Three SRM high-pass kernels used for the noise/residual stream (Zhou et al. style)."""
    k1 = np.array(
        [[0, 0, 0, 0, 0],
         [0, -1, 2, -1, 0],
         [0, 2, -4, 2, 0],
         [0, -1, 2, -1, 0],
         [0, 0, 0, 0, 0]], dtype=np.float32
    )
    k2 = np.array(
        [[-1, 2, -2, 2, -1],
         [2, -6, 8, -6, 2],
         [-2, 8, -12, 8, -2],
         [2, -6, 8, -6, 2],
         [-1, 2, -2, 2, -1]], dtype=np.float32
    )
    k3 = np.array(
        [[0, 0, 0, 0, 0],
         [0, 0, 0, 0, 0],
         [0, 1, -2, 1, 0],
         [0, 0, 0, 0, 0],
         [0, 0, 0, 0, 0]], dtype=np.float32
    )
    ks = []
    for k in (k1, k2, k3):
        s = np.abs(k).sum()
        ks.append(k / (s if s > 0 else 1.0))
    weight = np.stack(ks, 0)[:, None, :, :]  # 3,1,5,5
    return torch.from_numpy(weight)


class MultiStreamExtractor(nn.Module):
    """RGB + SRM residual + log-FFT frequency streams, fused by channel concat.

    Streams are computed from the RGB face crop so every experiment sees the
    same geometric crop; only the representation changes.
    """

    def __init__(self):
        super().__init__()
        self.register_buffer("srm_weight", _srm_kernels())

    def _denorm(self, x: torch.Tensor) -> torch.Tensor:
        mean = IMAGENET_MEAN.to(x.device, x.dtype)
        std = IMAGENET_STD.to(x.device, x.dtype)
        return (x * std + mean).clamp(0, 1)

    def forward(self, x_rgb_norm: torch.Tensor) -> torch.Tensor:
        rgb = self._denorm(x_rgb_norm)
        # Residual / SRM on luminance
        y = 0.299 * rgb[:, 0:1] + 0.587 * rgb[:, 1:2] + 0.114 * rgb[:, 2:3]
        residual = F.conv2d(y, self.srm_weight.to(dtype=rgb.dtype), padding=2)
        residual = residual.tanh()
        # Frequency: log magnitude spectrum per RGB channel, min-max per-sample
        spec = torch.fft.fftshift(torch.fft.fft2(rgb, norm="ortho"), dim=(-2, -1))
        mag = torch.log1p(spec.abs())
        b, c, h, w = mag.shape
        mag_flat = mag.reshape(b, c, -1)
        mn = mag_flat.min(dim=-1, keepdim=True).values
        mx = mag_flat.max(dim=-1, keepdim=True).values
        mag = ((mag_flat - mn) / (mx - mn + 1e-6)).reshape(b, c, h, w)
        fused = torch.cat([rgb, residual, mag], dim=1)  # B,9,H,W
        return fused


class FeatureFusion(nn.Module):
    """Project 9-channel fused tensor with an adapted ImageNet stem conv."""

    def __init__(self, conv3: nn.Conv2d):
        super().__init__()
        conv9 = nn.Conv2d(
            9, conv3.out_channels, kernel_size=conv3.kernel_size,
            stride=conv3.stride, padding=conv3.padding, bias=False,
        )
        with torch.no_grad():
            w = conv3.weight  # out,3,k,k
            conv9.weight[:, 0:3].copy_(w)
            conv9.weight[:, 3:6].copy_(w)
            conv9.weight[:, 6:9].copy_(w)
            conv9.weight.mul_(1.0 / 3.0)
        self.conv = conv9

    def forward(self, x):
        return self.conv(x)


class ASPP(nn.Module):
    def __init__(self, in_ch: int, out_ch: int = 256, rates=(6, 12, 18)):
        super().__init__()
        self.branches = nn.ModuleList()
        self.branches.append(nn.Sequential(
            nn.Conv2d(in_ch, out_ch, 1, bias=False),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
        ))
        for r in rates:
            self.branches.append(nn.Sequential(
                nn.Conv2d(in_ch, out_ch, 3, padding=r, dilation=r, bias=False),
                nn.BatchNorm2d(out_ch),
                nn.ReLU(inplace=True),
            ))
        # GAP is N,C,1,1. BatchNorm2d in train mode requires N*H*W > 1 and
        # crashes on the last batch when batch_size==1. GroupNorm is safe.
        gn = 32 if out_ch % 32 == 0 else 1
        self.gap = nn.Sequential(
            nn.AdaptiveAvgPool2d(1),
            nn.Conv2d(in_ch, out_ch, 1, bias=False),
            nn.GroupNorm(gn, out_ch),
            nn.ReLU(inplace=True),
        )
        self.project = nn.Sequential(
            nn.Conv2d(out_ch * 5, out_ch, 1, bias=False),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
            nn.Dropout(0.1),
        )

    def forward(self, x):
        res = [b(x) for b in self.branches]
        gap = self.gap(x)
        gap = F.interpolate(gap, size=x.shape[-2:], mode="bilinear", align_corners=False)
        res.append(gap)
        return self.project(torch.cat(res, dim=1))


class DeepLabV3PlusHead(nn.Module):
    def __init__(self, high_ch: int, low_ch: int, num_classes: int = 1):
        super().__init__()
        self.aspp = ASPP(high_ch, 256)
        self.low_proj = nn.Sequential(
            nn.Conv2d(low_ch, 48, 1, bias=False),
            nn.BatchNorm2d(48),
            nn.ReLU(inplace=True),
        )
        self.decoder = nn.Sequential(
            nn.Conv2d(304, 256, 3, padding=1, bias=False),
            nn.BatchNorm2d(256),
            nn.ReLU(inplace=True),
            nn.Conv2d(256, 256, 3, padding=1, bias=False),
            nn.BatchNorm2d(256),
            nn.ReLU(inplace=True),
            nn.Conv2d(256, num_classes, 1),
        )

    def forward(self, high, low, out_size):
        high = self.aspp(high)
        high = F.interpolate(high, size=low.shape[-2:], mode="bilinear", align_corners=False)
        low = self.low_proj(low)
        x = torch.cat([high, low], dim=1)
        x = self.decoder(x)
        x = F.interpolate(x, size=out_size, mode="bilinear", align_corners=False)
        return x


class ClassificationHead(nn.Module):
    def __init__(self, in_ch: int, dropout: float = 0.5):
        super().__init__()
        self.pool = nn.AdaptiveAvgPool2d(1)
        self.fc = nn.Sequential(
            nn.Dropout(dropout),
            nn.Linear(in_ch, 1),
        )

    def forward(self, feat):
        z = self.pool(feat).flatten(1)
        return self.fc(z)


class EfficientNetBackbone(nn.Module):
    """Wrap torchvision EfficientNet-B7 and expose stride-4 / stride-16 / final maps."""

    def __init__(self, in_channels: int = 3, pretrained: bool = True, multi_stream: bool = False):
        super().__init__()
        weights = EfficientNet_B7_Weights.IMAGENET1K_V1 if pretrained else None
        net = efficientnet_b7(weights=weights)
        stem = net.features[0]  # Conv2dNormActivation
        self.stem_rest = nn.Sequential(*list(stem.children())[1:])  # BN + SiLU
        conv0 = stem[0]
        self.multi_stream = multi_stream
        self.stream = MultiStreamExtractor() if multi_stream else None
        if multi_stream:
            self.fusion = FeatureFusion(conv0)
            self.stem_conv = self.fusion.conv
        else:
            self.fusion = None
            self.stem_conv = conv0
        self.stages = nn.ModuleList(list(net.features.children())[1:])
        self.out_channels = 2560
        self.low_stage = 1   # features[2], typically stride 4
        self.high_stage = 4  # features[5], typically stride 16

    def forward(self, x):
        if self.multi_stream:
            x9 = self.stream(x)
            x = self.stem_conv(x9)
        else:
            x = self.stem_conv(x)
        x = self.stem_rest(x)
        low = None
        high = None
        for i, stage in enumerate(self.stages):
            x = stage(x)
            if i == self.low_stage:
                low = x
            if i == self.high_stage:
                high = x
        if high is None:
            high = x
        if low is None:
            low = x
        return {"final": x, "low": low, "high": high}


class VERITASModel(nn.Module):
    def __init__(self, backbone="efficientnet_b7", multi_stream=False, segmentation=False, pretrained=True):
        super().__init__()
        if backbone != "efficientnet_b7":
            raise ValueError("This study uses EfficientNet-B7 only")
        self.multi_stream = bool(multi_stream)
        self.segmentation = bool(segmentation)
        self.encoder = EfficientNetBackbone(
            in_channels=3, pretrained=pretrained, multi_stream=self.multi_stream
        )
        self.cls_head = ClassificationHead(self.encoder.out_channels)
        # torchvision EfficientNet-B7: features[2]=48ch stride4, features[5]=224ch stride16
        self.seg_head = DeepLabV3PlusHead(224, 48, 1) if self.segmentation else None

    def forward(self, x):
        feats = self.encoder(x)
        logits = self.cls_head(feats["final"])
        out = {"classification_logits": logits, "fake_probability": torch.sigmoid(logits)}
        if self.segmentation:
            seg_logits = self.seg_head(feats["high"], feats["low"], out_size=x.shape[-2:])
            out["segmentation_logits"] = seg_logits
            out["segmentation_prob"] = torch.sigmoid(seg_logits)
        return out


def build_model(exp_cfg: Dict[str, Any]) -> VERITASModel:
    m = VERITASModel(
        backbone=exp_cfg.get("backbone", "efficientnet_b7"),
        multi_stream=exp_cfg["multi_stream"],
        segmentation=exp_cfg["segmentation"],
        pretrained=CONFIG["use_pretrained"],
    ).to(DEVICE)
    if m.segmentation:
        was_training = m.training
        m.eval()
        with torch.no_grad():
            dummy = torch.zeros(1, 3, 64, 64, device=DEVICE)
            feats = m.encoder(dummy)
            low_ch, high_ch = feats["low"].shape[1], feats["high"].shape[1]
        if m.seg_head.low_proj[0].in_channels != low_ch or m.seg_head.aspp.branches[0][0].in_channels != high_ch:
            m.seg_head = DeepLabV3PlusHead(high_ch, low_ch, 1).to(DEVICE)
            print(f"Adapted DeepLabV3+ head to low_ch={low_ch} high_ch={high_ch}")
        if was_training:
            m.train()
    return m


print("Model factory ready. Variants:")
for _name, _cfg in EXPERIMENTS.items():
    print(f"  {_name}: multi_stream={_cfg['multi_stream']} segmentation={_cfg['segmentation']}")



## 17. Losses


In [ ]:
class DiceLoss(nn.Module):
    def __init__(self, eps: float = 1e-6):
        super().__init__()
        self.eps = eps

    def forward(self, logits, targets):
        probs = torch.sigmoid(logits)
        dims = (1, 2, 3)
        inter = (probs * targets).sum(dims)
        den = probs.sum(dims) + targets.sum(dims)
        dice = (2 * inter + self.eps) / (den + self.eps)
        return 1.0 - dice.mean()


class VERITASLoss(nn.Module):
    """L_total = λ_cls * L_cls + λ_seg * L_seg
    L_seg = BCEWithLogits + dice_weight * Dice  (documented: Dice helps sparse fake masks)
    """

    def __init__(self, lambda_cls=1.0, lambda_seg=1.0, dice_weight=1.0, segmentation=False):
        super().__init__()
        self.lambda_cls = float(lambda_cls)
        self.lambda_seg = float(lambda_seg)
        self.dice_weight = float(dice_weight)
        self.segmentation = bool(segmentation)
        self.bce_cls = nn.BCEWithLogitsLoss()
        self.bce_seg = nn.BCEWithLogitsLoss()
        self.dice = DiceLoss()

    def forward(self, outputs, labels, masks=None):
        l_cls = self.bce_cls(outputs["classification_logits"], labels)
        l_seg = labels.new_zeros(())
        if self.segmentation:
            if masks is None:
                raise ValueError("masks required for segmentation experiments")
            l_seg = self.bce_seg(outputs["segmentation_logits"], masks) + self.dice_weight * self.dice(
                outputs["segmentation_logits"], masks
            )
        total = self.lambda_cls * l_cls + (self.lambda_seg * l_seg if self.segmentation else 0.0)
        return {
            "total": total,
            "classification": l_cls.detach(),
            "segmentation": l_seg.detach() if torch.is_tensor(l_seg) else l_seg,
            "lambda_cls": self.lambda_cls,
            "lambda_seg": self.lambda_seg if self.segmentation else 0.0,
        }


print("Loss: L_total = lambda_cls * L_cls + lambda_seg * L_seg")


## 18. DataLoaders


In [ ]:
def make_loader(samples, shuffle, augment):
    ds = FaceCropDataset(samples, GEO, augment=augment)
    drop_last = bool(shuffle and len(samples) >= int(CONFIG["batch_size"]))
    return DataLoader(
        ds,
        batch_size=CONFIG["batch_size"],
        shuffle=shuffle,
        num_workers=CONFIG["num_workers"],
        pin_memory=torch.cuda.is_available(),
        collate_fn=collate_faces,
        drop_last=drop_last,
    )


TRAIN_LOADER = make_loader(TRAIN_SAMPLES, shuffle=True, augment=True)
VAL_LOADER = make_loader(VAL_SAMPLES, shuffle=False, augment=False)
TEST_LOADER = make_loader(TEST_SAMPLES, shuffle=False, augment=False)
print("batch_size", CONFIG["batch_size"], "train batches", len(TRAIN_LOADER))



## 14b. Preprocessing before / after samples


In [ ]:
def _to_uint8(img):
    if img.dtype == np.uint8:
        return img
    x = img.astype(np.float32)
    x = x - x.min()
    x = x / (x.max() + 1e-6)
    return (x * 255.0).clip(0, 255).astype(np.uint8)


def _vis_residual(residual):
    """Display-only stretch. Model still uses tanh(SRM); this only makes the noise visible."""
    x = residual.astype(np.float32)
    lim = float(np.percentile(np.abs(x), 99.0)) + 1e-6
    x = (x / (2.0 * lim) + 0.5).clip(0.0, 1.0)
    return (x * 255.0).astype(np.uint8)


def _raw_bbox_crop(image_rgb, bbox_xyxy):
    h, w = image_rgb.shape[:2]
    x1, y1, x2, y2 = [int(round(v)) for v in bbox_xyxy]
    x1, y1 = max(0, x1), max(0, y1)
    x2, y2 = min(w, x2), min(h, y2)
    return image_rgb[y1:y2, x1:x2].copy()


def _overlay_poly_mask(rgb, mask, color=(255, 40, 40)):
    ov = rgb.copy()
    m = mask > 0
    if m.any():
        ov[m] = (0.45 * ov[m] + 0.55 * np.array(color)).astype(np.uint8)
    return ov


def _stream_visuals(letterboxed_rgb_uint8):
    """Reproduce the three VERITAS streams in numpy for inspection (same ops as the model)."""
    rgb = letterboxed_rgb_uint8.astype(np.float32) / 255.0
    y = 0.299 * rgb[..., 0] + 0.587 * rgb[..., 1] + 0.114 * rgb[..., 2]
    kernels = _srm_kernels().numpy()  # 3,1,5,5
    residual_ch = []
    for k in kernels:
        residual_ch.append(cv2.filter2D(y, -1, k[0], borderType=cv2.BORDER_REFLECT))
    residual = np.tanh(np.stack(residual_ch, axis=-1))
    spec = np.fft.fftshift(np.fft.fft2(rgb, axes=(0, 1), norm="ortho"), axes=(0, 1))
    mag = np.log1p(np.abs(spec))
    mag = (mag - mag.min()) / (mag.max() - mag.min() + 1e-6)
    return {
        "rgb": letterboxed_rgb_uint8,
        "residual": _vis_residual(residual),
        "frequency": _to_uint8(mag),
    }


def _pick_showcase_images(samples, n_mixed=2, n_fake=1, n_real=1):
    by_img = defaultdict(list)
    for s in samples:
        by_img[s.image_id].append(s)
    mixed, fake_only, real_only = [], [], []
    for iid, recs in by_img.items():
        labs = {r.label for r in recs}
        if labs == {0, 1}:
            mixed.append(iid)
        elif labs == {1}:
            fake_only.append(iid)
        elif labs == {0}:
            real_only.append(iid)
    chosen = mixed[:n_mixed] + fake_only[:n_fake] + real_only[:n_real]
    if not chosen:
        chosen = list(by_img.keys())[: max(1, n_mixed)]
    return [(iid, by_img[iid]) for iid in chosen]


def show_full_image_before_after(image_id, recs, save_prefix="preproc_image"):
    rgb = load_rgb(recs[0].image_path)
    before = rgb.copy()
    after = rgb.copy()
    for s in recs:
        x1, y1, x2, y2 = [int(round(v)) for v in s.bbox_xyxy]
        color = (220, 40, 40) if s.is_fake else (40, 180, 70)
        cv2.rectangle(after, (x1, y1), (x2, y2), color, 2)
        cv2.putText(
            after,
            f"face {s.face_id} {CATEGORY_NAME[s.label]}",
            (x1, max(12, y1 - 6)),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.5,
            color,
            2,
            cv2.LINE_AA,
        )
        for poly in s.polygon:
            cv2.polylines(after, [np.array(poly, dtype=np.int32)], True, color, 2)
    fig, ax = plt.subplots(1, 2, figsize=(12, 6))
    ax[0].imshow(before)
    ax[0].set_title(f"BEFORE  image_id={image_id}\nraw OpenForensics frame")
    ax[1].imshow(after)
    ax[1].set_title("AFTER annotation overlay\nboxes + face IDs + polygons (red=fake, green=real)")
    for a in ax:
        a.axis("off")
    fig.tight_layout()
    out = PATHS["working"] / "debug" / f"{save_prefix}_{image_id}.png"
    fig.savefig(out, bbox_inches="tight", dpi=130)
    plt.show()
    plt.close(fig)
    print("saved", out)


def show_face_preprocess_before_after(s: FaceSample, rgb=None):
    rgb = load_rgb(s.image_path) if rgb is None else rgb
    raw_crop = _raw_bbox_crop(rgb, s.bbox_xyxy)
    letterboxed, meta = GEO.apply_image(rgb, s.bbox_xyxy)
    model_polys = GEO.polygon_to_model(s.polygon, meta)
    mask = GEO.rasterize(model_polys, fake=s.is_fake)
    streams = _stream_visuals(letterboxed)

    # original-space mask projected from the model mask (checks invertibility of the transform)
    orig_h, orig_w = rgb.shape[:2]
    mask_on_original = GEO.mask_to_original(mask.astype(np.float32), meta, orig_h, orig_w)

    mean = np.array([0.485, 0.456, 0.406], dtype=np.float32)
    std = np.array([0.229, 0.224, 0.225], dtype=np.float32)
    normed = letterboxed.astype(np.float32) / 255.0
    normed = (normed - mean) / std

    fig, axes = plt.subplots(2, 4, figsize=(16, 8))
    titles_imgs = [
        (axes[0, 0], raw_crop, f"1. BEFORE letterbox\nraw bbox crop {raw_crop.shape[1]}x{raw_crop.shape[0]}"),
        (axes[0, 1], letterboxed, f"2. AFTER letterbox\nmodel RGB {GEO.size}x{GEO.size}  scale={meta['scale']:.3f}"),
        (axes[0, 2], _overlay_poly_mask(letterboxed, mask), f"3. AFTER mask rasterize\n{CATEGORY_NAME[s.label]}  mask_px={int(mask.sum())}"),
        (axes[0, 3], _to_uint8(normed), "4. AFTER ImageNet normalize\n(visualized, actually mean/std scaled)"),
        (axes[1, 0], streams["rgb"], "5. Stream RGB"),
        (axes[1, 1], streams["residual"], "6. Stream residual / SRM"),
        (axes[1, 2], streams["frequency"], "7. Stream frequency (log |FFT|)"),
        (axes[1, 3], _overlay_poly_mask(rgb, (mask_on_original > 0.5).astype(np.uint8)), "8. Mask projected BACK\nonto original image"),
    ]
    for ax, im, title in titles_imgs:
        ax.imshow(im)
        ax.set_title(title, fontsize=9)
        ax.axis("off")
    fig.suptitle(
        f"Face-level preprocess  image_id={s.image_id}  face_id={s.face_id}  {CATEGORY_NAME[s.label]}\n"
        f"bbox={tuple(round(v,1) for v in s.bbox_xyxy)}  pad(left,top)=({meta['left']:.1f},{meta['top']:.1f})",
        fontsize=11,
    )
    fig.tight_layout()
    out = PATHS["working"] / "debug" / f"preproc_face_{s.image_id}_{s.face_id}.png"
    fig.savefig(out, bbox_inches="tight", dpi=130)
    plt.show()
    plt.close(fig)
    print("saved", out)
    print(
        f"  crop {raw_crop.shape[1]}x{raw_crop.shape[0]} -> letterbox {GEO.size} "
        f"scale={meta['scale']:.4f} pad=({meta['left']:.1f},{meta['top']:.1f}) "
        f"label={CATEGORY_NAME[s.label]} mask_sum={int(mask.sum())}"
    )
    return out


def show_preprocessing_gallery(samples):
    showcases = _pick_showcase_images(samples)
    print("Preprocessing gallery: mixed/fake/real source images =", [iid for iid, _ in showcases])
    saved = []
    for iid, recs in showcases:
        show_full_image_before_after(iid, recs)
        rgb = load_rgb(recs[0].image_path)
        # one real and one fake from this image when available
        ordered = sorted(recs, key=lambda s: (-int(s.is_fake), s.face_id))
        for s in ordered[:2]:
            saved.append(show_face_preprocess_before_after(s, rgb=rgb))
    print("Wrote preprocessing figures under", PATHS["working"] / "debug")
    return saved


PREPROC_FIGS = show_preprocessing_gallery(TRAIN_SAMPLES)
try:
    from IPython.display import display, Image as IPyImage, Markdown
    debug_dir = PATHS["working"] / "debug"
    for p in list(sorted(debug_dir.glob("preproc_image_*.png"))) + list(sorted(debug_dir.glob("preproc_face_*.png"))):
        display(Markdown(f"### {p.stem}"))
        display(IPyImage(filename=str(p)))
except Exception as e:
    print("IPython display skipped (normal outside Jupyter):", e)


## 19–22. Training, validation, metrics, checkpointing


In [ ]:
def classification_metrics(y_true, y_pred):
    y_true = np.asarray(y_true).astype(int)
    y_pred = np.asarray(y_pred).astype(int)
    cm = confusion_matrix(y_true, y_pred, labels=[0, 1])
    tn, fp, fn, tp = cm.ravel()
    return {
        "accuracy": float(accuracy_score(y_true, y_pred)),
        "f1": float(f1_score(y_true, y_pred, zero_division=0)),
        "precision": float(precision_score(y_true, y_pred, zero_division=0)),
        "recall": float(recall_score(y_true, y_pred, zero_division=0)),
        "tp": int(tp), "tn": int(tn), "fp": int(fp), "fn": int(fn),
        "confusion_matrix": cm.tolist(),
        "n": int(len(y_true)),
    }


def mask_metrics_from_tensors(probs, targets, labels, thresh=0.5):
    """Face-level IoU / Dice.
    Policy: both-empty => IoU=Dice=1.
    Also report fake-only averages so real zero-masks cannot inflate scores.
    """
    pb = (probs > thresh)
    tb = (targets > 0.5)
    b = pb.shape[0]
    ious, dices, ious_fake, dices_fake = [], [], [], []
    for i in range(b):
        inter = (pb[i] & tb[i]).sum().item()
        union = (pb[i] | tb[i]).sum().item()
        iou = 1.0 if union == 0 else inter / union
        ps = pb[i].sum().item()
        ts = tb[i].sum().item()
        dice = 1.0 if (ps + ts) == 0 else (2 * inter) / (ps + ts)
        ious.append(iou)
        dices.append(dice)
        if int(labels[i]) == 1:
            ious_fake.append(iou)
            dices_fake.append(dice)
    return {
        "iou_mean_all": float(np.mean(ious) if ious else float("nan")),
        "dice_mean_all": float(np.mean(dices) if dices else float("nan")),
        "miou_fake": float(np.mean(ious_fake) if ious_fake else float("nan")),
        "dice_fake": float(np.mean(dices_fake) if dices_fake else float("nan")),
        "per_face_iou": ious,
        "per_face_dice": dices,
    }


@torch.no_grad()
def evaluate_loader(model, loader, criterion, split_name: str):
    model.eval()
    y_true, y_pred, y_prob = [], [], []
    recs = []
    loss_sum = 0.0
    n = 0
    iou_all, dice_all, iou_fake, dice_fake = [], [], [], []
    for batch in loader:
        images = batch["image"].to(DEVICE, non_blocking=True)
        labels = batch["label"].to(DEVICE, non_blocking=True)
        masks = batch["mask"].to(DEVICE, non_blocking=True)
        outputs = model(images)
        losses = criterion(outputs, labels, masks if model.segmentation else None)
        bs = images.size(0)
        loss_sum += float(losses["total"].item()) * bs
        n += bs
        prob = outputs["fake_probability"].detach().cpu().numpy().reshape(-1)
        pred = (prob >= CONFIG["classification_threshold"]).astype(int)
        true = labels.detach().cpu().numpy().reshape(-1).astype(int)
        y_true.extend(true.tolist())
        y_pred.extend(pred.tolist())
        y_prob.extend(prob.tolist())
        seg_prob = None
        if model.segmentation and "segmentation_prob" in outputs:
            mm = mask_metrics_from_tensors(outputs["segmentation_prob"].detach().cpu(), masks.detach().cpu(), true)
            iou_all.extend(mm["per_face_iou"])
            dice_all.extend(mm["per_face_dice"])
            for i, lab in enumerate(true):
                if lab == 1:
                    iou_fake.append(mm["per_face_iou"][i])
                    dice_fake.append(mm["per_face_dice"][i])
            seg_prob = outputs["segmentation_prob"].detach().cpu()
        for i in range(bs):
            rec = {
                "image_id": int(batch["image_id"][i]),
                "face_id": int(batch["face_id"][i]),
                "ann_id": int(batch["ann_id"][i]),
                "bbox": batch["bbox"][i].tolist(),
                "ground_truth_label": int(true[i]),
                "predicted_label": int(pred[i]),
                "fake_probability": float(prob[i]),
                "split": split_name,
            }
            if model.segmentation:
                rec["iou"] = mm["per_face_iou"][i]
                rec["dice"] = mm["per_face_dice"][i]
            recs.append(rec)
        del images, labels, masks, outputs
    metrics = classification_metrics(y_true, y_pred)
    metrics["loss"] = loss_sum / max(n, 1)
    if iou_all:
        metrics["iou_all"] = float(np.mean(iou_all))
        metrics["dice_all"] = float(np.mean(dice_all))
        metrics["miou"] = float(np.mean(iou_fake) if iou_fake else float("nan"))
        metrics["dice"] = float(np.mean(dice_fake) if dice_fake else float("nan"))
        metrics["miou_note"] = "miou/dice are macro averages over FAKE faces only; iou_all/dice_all include real zero-masks (both-empty=1)"
    else:
        metrics["iou_all"] = metrics["dice_all"] = metrics["miou"] = metrics["dice"] = None
    return metrics, recs


def save_checkpoint(path, model, optimizer, scheduler, epoch, best_metric, exp_name, exp_cfg, loss_cfg):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    torch.save(
        {
            "epoch": epoch,
            "model_state_dict": model.state_dict(),
            "optimizer_state_dict": optimizer.state_dict() if optimizer is not None else None,
            "scheduler_state_dict": scheduler.state_dict() if scheduler is not None else None,
            "best_metric": best_metric,
            "experiment_name": exp_name,
            "experiment_config": exp_cfg,
            "training_config": {k: v for k, v in CONFIG.items() if k != "loss_configs"},
            "loss_config": loss_cfg,
        },
        path,
    )


def load_checkpoint_if_any(path, model, optimizer=None, scheduler=None):
    path = Path(path)
    if not CONFIG["resume"] or not path.exists():
        return 0, None
    try:
        ckpt = torch.load(path, map_location=DEVICE, weights_only=False)
    except TypeError:
        ckpt = torch.load(path, map_location=DEVICE)
    model.load_state_dict(ckpt["model_state_dict"], strict=False)
    if optimizer is not None and ckpt.get("optimizer_state_dict"):
        try:
            optimizer.load_state_dict(ckpt["optimizer_state_dict"])
        except Exception as e:
            print("optimizer state not loaded:", e)
    if scheduler is not None and ckpt.get("scheduler_state_dict"):
        try:
            scheduler.load_state_dict(ckpt["scheduler_state_dict"])
        except Exception:
            pass
    print("Resumed", path, "epoch", ckpt.get("epoch"))
    return int(ckpt.get("epoch", 0)), ckpt.get("best_metric")


def measure_compute(model, loader, n_warmup=None, n_repeat=None):
    n_warmup = CONFIG["latency_warmup"] if n_warmup is None else n_warmup
    n_repeat = CONFIG["latency_repeats"] if n_repeat is None else n_repeat
    model.eval()
    batch = next(iter(loader))
    images = batch["image"].to(DEVICE)
    # warmup
    with torch.no_grad():
        for _ in range(n_warmup):
            _ = model(images)
    if torch.cuda.is_available():
        torch.cuda.reset_peak_memory_stats()
        torch.cuda.synchronize()
    times = []
    with torch.no_grad():
        for _ in range(n_repeat):
            if torch.cuda.is_available():
                torch.cuda.synchronize()
            t0 = time.perf_counter()
            _ = model(images)
            if torch.cuda.is_available():
                torch.cuda.synchronize()
            times.append((time.perf_counter() - t0) * 1000.0 / images.size(0))
    peak = None
    if torch.cuda.is_available():
        peak = torch.cuda.max_memory_allocated() / (1024 ** 2)
    return {
        "mean_latency_ms_per_face": float(np.mean(times)),
        "median_latency_ms_per_face": float(np.median(times)),
        "std_latency_ms_per_face": float(np.std(times)),
        "peak_vram_mb": None if peak is None else float(peak),
        "batch_size": int(images.size(0)),
        "input_resolution": int(images.shape[-1]),
        "device": str(DEVICE),
        "device_kind": "cuda" if torch.cuda.is_available() else "cpu",
    }


def train_experiment(exp_name, exp_cfg, train_loader, val_loader, loss_cfg=None):
    loss_cfg = loss_cfg or {"name": "default", "lambda_cls": CONFIG["lambda_cls"], "lambda_seg": CONFIG["lambda_seg"]}
    print("=" * 72)
    print("TRAIN", exp_name, exp_cfg, loss_cfg)
    model = build_model(exp_cfg)
    criterion = VERITASLoss(
        lambda_cls=loss_cfg["lambda_cls"],
        lambda_seg=loss_cfg["lambda_seg"],
        dice_weight=CONFIG["dice_weight"],
        segmentation=exp_cfg["segmentation"],
    )
    optimizer = torch.optim.AdamW(model.parameters(), lr=CONFIG["learning_rate"], weight_decay=CONFIG["weight_decay"])
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=max(CONFIG["epochs"], 1))
    ckpt_best = PATHS["working"] / "checkpoints" / f"{exp_name}_best.pt"
    start_epoch, best_metric = load_checkpoint_if_any(ckpt_best, model, optimizer, scheduler)
    best_metric = -1.0 if best_metric is None else float(best_metric)
    scaler_enabled = bool(CONFIG["use_amp"] and DEVICE.type == "cuda")
    try:
        scaler = torch.amp.GradScaler("cuda", enabled=scaler_enabled)
        autocast_ctx = lambda: torch.amp.autocast("cuda", enabled=scaler_enabled)
    except Exception:
        scaler = torch.cuda.amp.GradScaler(enabled=scaler_enabled)
        autocast_ctx = lambda: torch.cuda.amp.autocast(enabled=scaler_enabled)
    history = []
    if CONFIG["skip_training"]:
        print("skip_training=True")
        return model, {"history": history, "best_metric": best_metric}

    accum = max(int(CONFIG["gradient_accumulation"]), 1)
    for epoch in range(start_epoch, CONFIG["epochs"]):
        model.train()
        running = 0.0
        seen = 0
        optimizer.zero_grad(set_to_none=True)
        pbar = tqdm(train_loader, desc=f"{exp_name} ep{epoch+1}/{CONFIG['epochs']}")
        for step, batch in enumerate(pbar):
            images = batch["image"].to(DEVICE, non_blocking=True)
            labels = batch["label"].to(DEVICE, non_blocking=True)
            masks = batch["mask"].to(DEVICE, non_blocking=True)
            with autocast_ctx():
                outputs = model(images)
                losses = criterion(outputs, labels, masks if model.segmentation else None)
                loss = losses["total"] / accum
            scaler.scale(loss).backward()
            if (step + 1) % accum == 0 or (step + 1) == len(train_loader):
                scaler.step(optimizer)
                scaler.update()
                optimizer.zero_grad(set_to_none=True)
            bs = images.size(0)
            running += float(losses["total"].item()) * bs
            seen += bs
            pbar.set_postfix(
                loss=f"{losses['total'].item():.4f}",
                cls=f"{float(losses['classification']):.4f}",
                seg=f"{float(losses['segmentation']):.4f}",
            )
            del images, labels, masks, outputs, loss
        scheduler.step()
        train_loss = running / max(seen, 1)
        val_metrics, _ = evaluate_loader(model, val_loader, criterion, "val")
        row = {
            "epoch": epoch + 1,
            "train_loss": train_loss,
            "val_loss": val_metrics["loss"],
            "val_f1": val_metrics["f1"],
            "val_acc": val_metrics["accuracy"],
            "lambda_cls": loss_cfg["lambda_cls"],
            "lambda_seg": loss_cfg["lambda_seg"] if exp_cfg["segmentation"] else 0.0,
            "val_miou_fake": val_metrics.get("miou"),
        }
        history.append(row)
        print(row)
        with open(PATHS["working"] / "logs" / f"{exp_name}_history.json", "w", encoding="utf-8") as f:
            json.dump(history, f, indent=2)
        save_checkpoint(PATHS["working"] / "checkpoints" / f"{exp_name}_last.pt", model, optimizer, scheduler, epoch + 1, best_metric, exp_name, exp_cfg, loss_cfg)
        if val_metrics["f1"] >= best_metric:
            best_metric = val_metrics["f1"]
            save_checkpoint(ckpt_best, model, optimizer, scheduler, epoch + 1, best_metric, exp_name, exp_cfg, loss_cfg)
            print("  saved best", ckpt_best, "F1", best_metric)
    # reload best
    if ckpt_best.exists():
        load_checkpoint_if_any(ckpt_best, model)
    return model, {"history": history, "best_metric": best_metric, "loss_config": loss_cfg}


## 23. Automated tests (multi-face, matching, masks, transforms)


In [ ]:
def _synthetic_image(w=160, h=120, faces=None):
    """faces: list of dicts {bbox_xyxy, label, polygon?} drawn into a canvas."""
    img = np.zeros((h, w, 3), dtype=np.uint8)
    img[:] = (30, 30, 40)
    anns = []
    for i, f in enumerate(faces or []):
        x1, y1, x2, y2 = [int(v) for v in f["bbox"]]
        color = (200, 40, 40) if f["label"] == 1 else (40, 180, 80)
        cv2.rectangle(img, (x1, y1), (x2, y2), color, -1)
        poly = f.get("polygon")
        if poly is None:
            poly = [[(x1, y1), (x2, y1), (x2, y2), (x1, y2)]]
        anns.append({"face_id": i, "bbox": [x1, y1, x2, y2], "label": f["label"], "polygon": poly})
    return img, anns


def _gt_samples_from_anns(image_rgb, anns, image_id=0):
    h, w = image_rgb.shape[:2]
    samples = []
    boxes = [tuple(a["bbox"]) for a in anns]
    order = stable_face_id_order(boxes)
    for face_id, idx in enumerate(order):
        a = anns[idx]
        samples.append(
            FaceSample(
                image_id=image_id,
                face_id=face_id,
                bbox_xyxy=tuple(map(float, a["bbox"])),
                label=int(a["label"]),
                file_name="synthetic.jpg",
                image_path="",
                polygon=a["polygon"],
                ann_id=idx,
            )
        )
    return samples


def _masks_for(image_rgb, samples):
    h, w = image_rgb.shape[:2]
    geo = GeometryTransform(64, 0.0)
    out = []
    for s in samples:
        dummy_img, meta = geo.apply_image(image_rgb, s.bbox_xyxy)
        mp = geo.polygon_to_model(s.polygon, meta)
        out.append(geo.rasterize(mp, fake=s.is_fake))
    return out


def run_unit_tests():
    results = []

    def check(name, cond, detail=""):
        results.append({"test": name, "pass": bool(cond), "detail": detail})
        print(("PASS" if cond else "FAIL"), name, detail)

    # Test 1 — one real face
    img, anns = _synthetic_image(faces=[{"bbox": [20, 20, 70, 80], "label": 0}])
    samples = _gt_samples_from_anns(img, anns)
    masks = _masks_for(img, samples)
    check("T1_one_real_count", len(samples) == 1)
    check("T1_one_real_label", samples[0].label == 0)
    check("T1_one_real_mask_zero", masks[0].sum() == 0)

    # Test 2 — one fake face
    img, anns = _synthetic_image(faces=[{"bbox": [20, 20, 80, 90], "label": 1}])
    samples = _gt_samples_from_anns(img, anns)
    masks = _masks_for(img, samples)
    check("T2_one_fake_count", len(samples) == 1)
    check("T2_one_fake_label", samples[0].label == 1)
    check("T2_one_fake_mask_nonzero", masks[0].sum() > 0)

    # Test 3 — three real faces
    img, anns = _synthetic_image(faces=[
        {"bbox": [5, 5, 35, 40], "label": 0},
        {"bbox": [50, 10, 90, 50], "label": 0},
        {"bbox": [100, 20, 150, 70], "label": 0},
    ])
    samples = _gt_samples_from_anns(img, anns)
    masks = _masks_for(img, samples)
    check("T3_three_real_count", len(samples) == 3)
    check("T3_three_real_labels", all(s.label == 0 for s in samples))
    check("T3_three_real_masks", all(m.sum() == 0 for m in masks))

    # Test 4 — mixed real/fake/real
    img, anns = _synthetic_image(faces=[
        {"bbox": [5, 5, 40, 50], "label": 0},
        {"bbox": [55, 8, 100, 60], "label": 1},
        {"bbox": [110, 12, 155, 58], "label": 0},
    ])
    samples = _gt_samples_from_anns(img, anns)
    masks = _masks_for(img, samples)
    labs = [s.label for s in samples]
    check("T4_mixed_labels_set", sorted(labs) == [0, 0, 1])
    fake_idx = [i for i, s in enumerate(samples) if s.label == 1]
    real_idx = [i for i, s in enumerate(samples) if s.label == 0]
    check("T4_fake_mask", all(masks[i].sum() > 0 for i in fake_idx))
    check("T4_real_mask_zero", all(masks[i].sum() == 0 for i in real_idx))

    # Test 5 — multiple fake faces, independent polygons
    img, anns = _synthetic_image(faces=[
        {"bbox": [5, 5, 45, 55], "label": 1},
        {"bbox": [70, 10, 120, 65], "label": 1},
    ])
    samples = _gt_samples_from_anns(img, anns)
    masks = _masks_for(img, samples)
    check("T5_two_fake", len(samples) == 2 and all(s.label == 1 for s in samples))
    check("T5_independent_masks", masks[0].sum() > 0 and masks[1].sum() > 0)

    # Test 6 — detector order differs from annotation order
    ann_boxes = [(10, 10, 40, 40), (80, 15, 120, 60)]
    det_boxes = [(78, 14, 122, 61), (11, 11, 39, 39)]  # reversed
    m = match_faces_to_annotations(det_boxes, ann_boxes, 0.3)
    pair_map = {p["det_index"]: p["ann_index"] for p in m["pairs"]}
    check("T6_reversed_match", pair_map.get(0) == 1 and pair_map.get(1) == 0, str(pair_map))
    check("T6_no_unmatched", m["unmatched_det"] == [] and m["unmatched_ann"] == [])

    # Test 7 — no faces
    empty = []
    try:
        empty = FACE_DETECTOR.detect(np.zeros((64, 64, 3), dtype=np.uint8))
    except Exception as e:
        empty = []
        print("detector on empty image raised (treated as no faces):", e)
    check("T7_no_faces_graceful", isinstance(empty, list))
    m0 = match_faces_to_annotations([], [], 0.3)
    check("T7_empty_match", m0["pairs"] == [] and m0["unmatched_det"] == [] and m0["unmatched_ann"] == [])

    # Test 8 — polygon lands in the correct crop location after resize
    poly = [[(30, 30), (70, 30), (70, 70), (30, 70)]]
    img, anns = _synthetic_image(w=200, h=150, faces=[{"bbox": [20, 20, 80, 80], "label": 1, "polygon": poly}])
    geo = GeometryTransform(64, 0.0)
    crop, meta = geo.apply_image(img, anns[0]["bbox"])
    mp = geo.polygon_to_model(poly, meta)
    mask = geo.rasterize(mp, True)
    # centroid of mask should be near transformed polygon centroid
    ys, xs = np.where(mask > 0)
    check("T8_mask_nonempty", len(xs) > 0)
    if len(xs) > 0:
        cx, cy = xs.mean(), ys.mean()
        tpoly = np.array(mp[0], dtype=np.float32)
        tcx, tcy = tpoly[:, 0].mean(), tpoly[:, 1].mean()
        dist = ((cx - tcx) ** 2 + (cy - tcy) ** 2) ** 0.5
        check("T8_centroid_alignment", dist < 8.0, f"dist={dist:.2f}")

    # Test 9 — real face must not inherit a neighbouring fake polygon
    img, anns = _synthetic_image(faces=[
        {"bbox": [5, 5, 50, 60], "label": 0},
        {"bbox": [70, 5, 130, 70], "label": 1},
    ])
    samples = _gt_samples_from_anns(img, anns)
    masks = _masks_for(img, samples)
    real = [m for s, m in zip(samples, masks) if s.label == 0][0]
    fake = [m for s, m in zip(samples, masks) if s.label == 1][0]
    check("T9_no_contamination", real.sum() == 0 and fake.sum() > 0)

    # Extra: mixed case from real dataset if available
    mixed_ids = ANN_DF.groupby("image_id")["category_id"].nunique()
    mixed_ids = mixed_ids[mixed_ids > 1].index.tolist()
    if mixed_ids and TEST_SAMPLES:
        sid = None
        by_img = defaultdict(list)
        for s in TRAIN_SAMPLES + VAL_SAMPLES + TEST_SAMPLES:
            by_img[s.image_id].append(s)
        for iid, recs in by_img.items():
            labs = {r.label for r in recs}
            if labs == {0, 1} and len(recs) >= 2:
                sid = iid
                mixed_recs = recs
                break
        if sid is not None:
            check("T_real_mixed_exists", True, f"image_id={sid} n={len(mixed_recs)}")
            check("T_real_mixed_independent_labels", len({(r.face_id, r.label) for r in mixed_recs}) == len(mixed_recs))
        else:
            check("T_real_mixed_exists", False, "no mixed image in current subset")

    n_pass = sum(1 for r in results if r["pass"])
    print(f"\n{n_pass}/{len(results)} tests passed")
    with open(PATHS["working"] / "logs" / "unit_tests.json", "w", encoding="utf-8") as f:
        json.dump(results, f, indent=2)
    return results


TEST_RESULTS = run_unit_tests()


## 24. Multi-face inference


In [ ]:
FACE_CACHE_PATH = PATHS["working"] / "face_metadata.json"
_DETECTOR_CACHE = {"meta": {}, "by_path": {}}


def _image_fingerprint(path: Path) -> str:
    st = path.stat()
    return f"{path.name}:{st.st_size}:{int(st.st_mtime)}"


def _cache_valid(meta) -> bool:
    if not meta:
        return False
    if meta.get("detector") != FACE_DETECTOR.name:
        return False
    if meta.get("min_confidence") != CONFIG["face_score_threshold"]:
        return False
    return True


def load_face_cache():
    global _DETECTOR_CACHE
    if FACE_CACHE_PATH.exists():
        with open(FACE_CACHE_PATH, "r", encoding="utf-8") as f:
            _DETECTOR_CACHE = json.load(f)
        if not _cache_valid(_DETECTOR_CACHE.get("meta", {})):
            print("Face cache incompatible with current detector; rebuilding")
            _DETECTOR_CACHE = {"meta": {}, "by_path": {}}
        else:
            print("Loaded face cache with", len(_DETECTOR_CACHE.get("by_path", {})), "images")


def save_face_cache():
    _DETECTOR_CACHE["meta"] = {
        "detector": FACE_DETECTOR.name,
        "min_confidence": CONFIG["face_score_threshold"],
        "version": 1,
    }
    with open(FACE_CACHE_PATH, "w", encoding="utf-8") as f:
        json.dump(_DETECTOR_CACHE, f)


load_face_cache()


def detect_faces_cached(image_path: str, image_rgb=None):
    path = Path(image_path)
    fp = _image_fingerprint(path) if path.exists() else "mem"
    rec = _DETECTOR_CACHE.get("by_path", {}).get(str(path))
    if rec and rec.get("fingerprint") == fp:
        return rec["faces"]
    if image_rgb is None:
        image_rgb = load_rgb(str(path))
    faces = FACE_DETECTOR.detect(image_rgb)
    _DETECTOR_CACHE.setdefault("by_path", {})[str(path)] = {"fingerprint": fp, "faces": faces}
    return faces


@torch.no_grad()
def predict_multi_face(image_path, model=None, return_masks=True, gt_by_image=None):
    """Detect every face independently. Never collapse to one image-level label."""
    image_rgb = load_rgb(image_path)
    h, w = image_rgb.shape[:2]
    faces = detect_faces_cached(image_path, image_rgb)
    results = {"image_id": Path(image_path).stem, "image_path": str(image_path), "faces": []}
    if not faces:
        return results
    model = model or GLOBAL_MODEL
    model.eval()
    geo = GEO
    mean = np.array([0.485, 0.456, 0.406], dtype=np.float32)
    std = np.array([0.229, 0.224, 0.225], dtype=np.float32)
    tensors = []
    metas = []
    for f in faces:
        crop, meta = geo.apply_image(image_rgb, f["bbox"])
        x = crop.astype(np.float32) / 255.0
        x = (x - mean) / std
        tensors.append(torch.from_numpy(np.transpose(x, (2, 0, 1)).copy()))
        metas.append(meta)
    batch = torch.stack(tensors, 0).to(DEVICE)
    outputs = model(batch)
    probs = outputs["fake_probability"].detach().cpu().numpy().reshape(-1)
    seg = outputs.get("segmentation_prob")
    if seg is not None:
        seg = seg.detach().cpu().numpy()
    for i, f in enumerate(faces):
        p = float(probs[i])
        label = "fake" if p >= CONFIG["classification_threshold"] else "real"
        mask_full = None
        if return_masks and seg is not None and label == "fake":
            mask_full = geo.mask_to_original(seg[i, 0], metas[i], h, w)
        results["faces"].append(
            {
                "face_id": int(f["face_id"]),
                "bbox": [float(x) for x in f["bbox"]],
                "confidence": float(f.get("confidence", 0.0)),
                "label": label,
                "fake_probability": p,
                "mask": mask_full,
            }
        )
    return results


GLOBAL_MODEL = None
print("predict_multi_face ready")


## 25. Full-image visualization and debug_image


In [ ]:
def annotations_for_image_id(image_id: int) -> List[FaceSample]:
    recs = [s for s in TRAIN_SAMPLES + VAL_SAMPLES + TEST_SAMPLES if s.image_id == int(image_id)]
    if recs:
        return recs
    return records_from_annotations(ANN_DF, [int(image_id)])


def debug_image(image_path, model=None, image_id=None, save=True):
    image_rgb = load_rgb(image_path)
    pred = predict_multi_face(image_path, model=model)
    # GT from filename match if image_id unknown
    if image_id is None:
        stem = Path(image_path).name
        hits = ANN_DF[ANN_DF["file_name"].astype(str).str.endswith(stem)]["image_id"].unique().tolist()
        image_id = int(hits[0]) if hits else None
    gt = annotations_for_image_id(image_id) if image_id is not None else []

    fig, axes = plt.subplots(2, 4, figsize=(16, 8))
    axes = axes.ravel()
    axes[0].imshow(image_rgb)
    axes[0].set_title("1. Original")
    vis_det = image_rgb.copy()
    for f in pred["faces"]:
        x1, y1, x2, y2 = [int(round(v)) for v in f["bbox"]]
        cv2.rectangle(vis_det, (x1, y1), (x2, y2), (0, 200, 255), 2)
        cv2.putText(vis_det, f"id {f['face_id']}", (x1, y1 - 4), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 200, 255), 1)
    axes[1].imshow(vis_det)
    axes[1].set_title("2–3. Detected boxes + IDs")

    vis_gt = image_rgb.copy()
    for s in gt:
        x1, y1, x2, y2 = [int(round(v)) for v in s.bbox_xyxy]
        color = (220, 40, 40) if s.is_fake else (40, 180, 70)
        cv2.rectangle(vis_gt, (x1, y1), (x2, y2), color, 2)
        cv2.putText(vis_gt, f"{s.face_id}:{CATEGORY_NAME[s.label]}", (x1, max(0, y1 - 4)), cv2.FONT_HERSHEY_SIMPLEX, 0.45, color, 1)
        for poly in s.polygon:
            pts = np.array(poly, dtype=np.int32)
            cv2.polylines(vis_gt, [pts], True, color, 1)
    axes[2].imshow(vis_gt)
    axes[2].set_title("4–5. GT labels + polygons")

    gt_overlay = image_rgb.copy()
    if gt:
        acc = np.zeros(image_rgb.shape[:2], dtype=np.float32)
        for s in gt:
            crop, meta = GEO.apply_image(image_rgb, s.bbox_xyxy)
            mp = GEO.polygon_to_model(s.polygon, meta)
            m = GEO.rasterize(mp, s.is_fake)
            acc = np.maximum(acc, GEO.mask_to_original(m.astype(np.float32), meta, *image_rgb.shape[:2]))
        gt_overlay[acc > 0.5] = (0.4 * gt_overlay[acc > 0.5] + 0.6 * np.array([220, 40, 40])).astype(np.uint8)
    axes[3].imshow(gt_overlay)
    axes[3].set_title("6. GT manipulation masks")

    vis_pred = image_rgb.copy()
    for f in pred["faces"]:
        x1, y1, x2, y2 = [int(round(v)) for v in f["bbox"]]
        color = (220, 40, 40) if f["label"] == "fake" else (40, 180, 70)
        cv2.rectangle(vis_pred, (x1, y1), (x2, y2), color, 2)
        cv2.putText(vis_pred, f"{f['face_id']} {f['label']} {f['fake_probability']:.2f}", (x1, max(0, y1 - 4)), cv2.FONT_HERSHEY_SIMPLEX, 0.4, color, 1)
    axes[4].imshow(vis_pred)
    axes[4].set_title("7. Predicted labels")

    pred_ov = image_rgb.copy()
    for f in pred["faces"]:
        if f.get("mask") is not None:
            m = f["mask"]
            pred_ov[m > 0.5] = (0.4 * pred_ov[m > 0.5] + 0.6 * np.array([255, 80, 0])).astype(np.uint8)
    axes[5].imshow(pred_ov)
    axes[5].set_title("8. Predicted masks")

    # matching panel
    if gt and pred["faces"]:
        m = match_faces_to_annotations(
            [tuple(f["bbox"]) for f in pred["faces"]],
            [s.bbox_xyxy for s in gt],
            CONFIG["match_iou_threshold"],
        )
        axes[6].axis("off")
        axes[6].set_title("Matching log")
        txt = json.dumps({k: m[k] for k in ("pairs", "unmatched_det", "unmatched_ann", "ambiguous")}, indent=2)[:800]
        axes[6].text(0, 1, txt, va="top", family="monospace", fontsize=8)
    else:
        axes[6].axis("off")
        axes[6].set_title("Matching log (no GT)")

    axes[7].axis("off")
    axes[7].set_title("Policy")
    axes[7].text(0, 1, "Unmatched annotation: not fabricated.\nUnmatched detection: unlabeled at train time.\nReal mask is always zeros.\nOne fake face never labels others.", va="top", fontsize=9)
    for ax in axes:
        if ax.has_data() or ax.get_title():
            pass
        ax.axis("off")
    fig.tight_layout()
    out = None
    if save:
        out = PATHS["working"] / "debug" / f"debug_{Path(image_path).stem}.png"
        fig.savefig(out, bbox_inches="tight")
        print("saved", out)
    plt.show()
    plt.close(fig)
    return pred


## 26. Error analysis helpers


In [ ]:
def error_analysis(records: List[Dict[str, Any]], title: str, n_show: int = 6):
    if not records:
        print("No records for error analysis")
        return
    df = pd.DataFrame(records)
    fp = df[(df.ground_truth_label == 0) & (df.predicted_label == 1)]
    fn = df[(df.ground_truth_label == 1) & (df.predicted_label == 0)]
    print(title, "FP", len(fp), "FN", len(fn), "N", len(df))
    report = {
        "n": len(df),
        "fp": int(len(fp)),
        "fn": int(len(fn)),
        "fp_examples": fp.head(n_show)[["image_id", "face_id", "fake_probability"]].to_dict("records"),
        "fn_examples": fn.head(n_show)[["image_id", "face_id", "fake_probability"]].to_dict("records"),
    }
    if "iou" in df.columns:
        fake = df[df.ground_truth_label == 1]
        report["fake_iou_mean"] = float(fake["iou"].mean()) if len(fake) else None
        report["worst_iou"] = fake.nsmallest(n_show, "iou")[["image_id", "face_id", "iou", "dice"]].to_dict("records") if len(fake) else []
    out = PATHS["working"] / "logs" / f"error_analysis_{title}.json"
    with open(out, "w", encoding="utf-8") as f:
        json.dump(report, f, indent=2)
    print("wrote", out)
    return report


def mcnemar_from_records(rec_a, rec_b):
    """Paired classification comparison on the same (image_id, face_id) keys."""
    key = lambda r: (int(r["image_id"]), int(r["face_id"]))
    ma = {key(r): r for r in rec_a}
    mb = {key(r): r for r in rec_b}
    common = sorted(set(ma) & set(mb))
    if not common:
        return {"error": "no paired samples"}
    n01 = n10 = 0
    for k in common:
        a_wrong = int(ma[k]["predicted_label"] != ma[k]["ground_truth_label"])
        b_wrong = int(mb[k]["predicted_label"] != mb[k]["ground_truth_label"])
        if a_wrong == 0 and b_wrong == 1:
            n01 += 1
        elif a_wrong == 1 and b_wrong == 0:
            n10 += 1
    n = n01 + n10
    # exact binomial McNemar
    if n == 0:
        p = 1.0
    else:
        p = float(stats.binomtest(min(n01, n10), n=n, p=0.5).pvalue)
    return {"n_paired": len(common), "n01_a_correct_b_wrong": n01, "n10_a_wrong_b_correct": n10, "pvalue": p}


def wilcoxon_iou(rec_a, rec_b):
    key = lambda r: (int(r["image_id"]), int(r["face_id"]))
    ma = {key(r): r for r in rec_a if "iou" in r}
    mb = {key(r): r for r in rec_b if "iou" in r}
    common = sorted(set(ma) & set(mb))
    diffs = [ma[k]["iou"] - mb[k]["iou"] for k in common]
    if len(diffs) < 10:
        return {"n": len(diffs), "note": "too few paired IoU values"}
    try:
        stat, p = stats.wilcoxon(diffs)
        return {"n": len(diffs), "stat": float(stat), "pvalue": float(p)}
    except Exception as e:
        return {"n": len(diffs), "error": str(e)}


## 27. Run experiments and write the results table


In [ ]:
def export_predictions(exp_name, records, model, loader):
    pred_dir = PATHS["working"] / "predictions"
    df = pd.DataFrame(records)
    csv_path = pred_dir / f"{exp_name}_predictions.csv"
    json_path = pred_dir / f"{exp_name}_predictions.json"
    df.drop(columns=["mask"] if "mask" in df.columns else [], errors="ignore").to_csv(csv_path, index=False)
    # JSON without bulky arrays
    slim = []
    for r in records:
        slim.append({k: v for k, v in r.items() if k not in {"mask", "ground_truth_mask"}})
    with open(json_path, "w", encoding="utf-8") as f:
        json.dump(slim, f)
    print("wrote", csv_path)
    return csv_path


def run_all_experiments():
    global GLOBAL_MODEL
    summary_rows = []
    all_test_records = {}
    models = {}

    for exp_name in CONFIG["experiments_to_run"]:
        exp_cfg = EXPERIMENTS[exp_name]
        try:
            model, train_info = train_experiment(exp_name, exp_cfg, TRAIN_LOADER, VAL_LOADER)
            criterion = VERITASLoss(
                CONFIG["lambda_cls"], CONFIG["lambda_seg"], CONFIG["dice_weight"], exp_cfg["segmentation"]
            )
            test_metrics, test_recs = evaluate_loader(model, TEST_LOADER, criterion, "test")
            compute = measure_compute(model, TEST_LOADER)
            export_predictions(exp_name, test_recs, model, TEST_LOADER)
            error_analysis(test_recs, exp_name)
            save_face_cache()
            row = {
                "experiment": exp_name,
                "multi_stream": exp_cfg["multi_stream"],
                "segmentation": exp_cfg["segmentation"],
                "accuracy": test_metrics["accuracy"],
                "f1": test_metrics["f1"],
                "precision": test_metrics["precision"],
                "recall": test_metrics["recall"],
                "tp": test_metrics["tp"], "tn": test_metrics["tn"], "fp": test_metrics["fp"], "fn": test_metrics["fn"],
                "miou_fake_only": test_metrics.get("miou"),
                "dice_fake_only": test_metrics.get("dice"),
                "iou_all_faces": test_metrics.get("iou_all"),
                "dice_all_faces": test_metrics.get("dice_all"),
                "mean_latency_ms": compute["mean_latency_ms_per_face"],
                "median_latency_ms": compute["median_latency_ms_per_face"],
                "peak_vram_mb": compute["peak_vram_mb"],
                "batch_size": compute["batch_size"],
                "input_resolution": compute["input_resolution"],
                "device": compute["device"],
                "status": "ok",
            }
            all_test_records[exp_name] = test_recs
            models[exp_name] = model
            GLOBAL_MODEL = model
            summary_rows.append(row)
            with open(PATHS["working"] / "logs" / f"{exp_name}_test_metrics.json", "w", encoding="utf-8") as f:
                json.dump({"metrics": test_metrics, "compute": compute, "config": exp_cfg}, f, indent=2)
        except Exception:
            traceback.print_exc()
            summary_rows.append(
                {
                    "experiment": exp_name,
                    "multi_stream": exp_cfg["multi_stream"],
                    "segmentation": exp_cfg["segmentation"],
                    "accuracy": "NOT RUN",
                    "f1": "NOT RUN",
                    "precision": "NOT RUN",
                    "recall": "NOT RUN",
                    "miou_fake_only": "N/A" if not exp_cfg["segmentation"] else "NOT RUN",
                    "dice_fake_only": "N/A" if not exp_cfg["segmentation"] else "NOT RUN",
                    "mean_latency_ms": "NOT RUN",
                    "peak_vram_mb": "NOT RUN",
                    "status": "failed",
                }
            )

        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    # Loss-weight sweeps on full VERITAS (optional)
    if CONFIG["run_loss_sweeps"] and "full_veritas" in EXPERIMENTS:
        for lc in CONFIG["loss_configs"]:
            name = f"full_veritas_loss_{lc['name']}"
            try:
                model, _ = train_experiment(name, EXPERIMENTS["full_veritas"], TRAIN_LOADER, VAL_LOADER, loss_cfg=lc)
                criterion = VERITASLoss(lc["lambda_cls"], lc["lambda_seg"], CONFIG["dice_weight"], True)
                test_metrics, test_recs = evaluate_loader(model, TEST_LOADER, criterion, "test")
                export_predictions(name, test_recs, model, TEST_LOADER)
                summary_rows.append(
                    {
                        "experiment": name,
                        "multi_stream": True,
                        "segmentation": True,
                        "accuracy": test_metrics["accuracy"],
                        "f1": test_metrics["f1"],
                        "miou_fake_only": test_metrics.get("miou"),
                        "dice_fake_only": test_metrics.get("dice"),
                        "lambda_cls": lc["lambda_cls"],
                        "lambda_seg": lc["lambda_seg"],
                        "status": "ok",
                    }
                )
            except Exception:
                traceback.print_exc()
                summary_rows.append({"experiment": name, "status": "failed"})

    summary = pd.DataFrame(summary_rows)
    out_csv = PATHS["working"] / "experiment_results.csv"
    summary.to_csv(out_csv, index=False)
    print("\n=== EXPERIMENT RESULTS ===")
    print(summary.to_string(index=False))
    print("saved", out_csv)

    stats_out = {}
    names = [r["experiment"] for r in summary_rows if r.get("status") == "ok"]
    if "baseline_effb7" in all_test_records and "full_veritas" in all_test_records:
        stats_out["mcnemar_baseline_vs_full"] = mcnemar_from_records(
            all_test_records["baseline_effb7"], all_test_records["full_veritas"]
        )
    if "seg_no_multistream" in all_test_records and "full_veritas" in all_test_records:
        stats_out["wilcoxon_iou_seg_vs_full"] = wilcoxon_iou(
            all_test_records["seg_no_multistream"], all_test_records["full_veritas"]
        )
    with open(PATHS["working"] / "logs" / "statistical_tests.json", "w", encoding="utf-8") as f:
        json.dump(stats_out, f, indent=2)
    print("statistical tests:", stats_out)
    return summary, models, all_test_records


SUMMARY_DF, TRAINED_MODELS, TEST_RECORDS = run_all_experiments()


## 28. Qualitative multi-face visualization on held-out images


In [ ]:
def pick_qualitative_images(n=4):
    # prefer mixed real/fake test images
    by_img = defaultdict(list)
    for s in TEST_SAMPLES:
        by_img[s.image_id].append(s)
    mixed, others = [], []
    for iid, recs in by_img.items():
        labs = {r.label for r in recs}
        (mixed if labs == {0, 1} else others).append(recs[0])
    chosen = (mixed + others)[:n]
    return chosen


qual = pick_qualitative_images(4)
model_vis = TRAINED_MODELS.get("full_veritas") or TRAINED_MODELS.get("baseline_effb7") or GLOBAL_MODEL
if model_vis is None and TRAINED_MODELS:
    model_vis = next(iter(TRAINED_MODELS.values()))

for s in qual:
    print("debug_image", s.image_path, "image_id", s.image_id, "n_faces_in_split_sample>=1")
    try:
        pred = debug_image(s.image_path, model=model_vis, image_id=s.image_id, save=True)
        print("predicted faces:", [(f["face_id"], f["label"], round(f["fake_probability"], 3)) for f in pred["faces"]])
    except Exception:
        traceback.print_exc()

print("\nWorking artifacts:")
for p in sorted(PATHS["working"].rglob("*")):
    if p.is_file():
        print(" ", p.relative_to(PATHS["working"]), p.stat().st_size)


## Notes for a full research run

1. Set `CONFIG["run_mode"] = "full"`.
2. Keep `image_size = 600` (EfficientNet-B7 native).
3. Attach dataset `nathanielescuro/veritas-openforensics-compiled`.
4. Enable GPU (T4/P100/H100). Reduce `batch_size` or raise `gradient_accumulation` on OOM.
5. Official OpenForensics splits: `split_protocol = "official_openforensics"`.
6. SOP 70/15/15 is the default and splits at **image** level so faces from one photo cannot leak across train/val/test.
7. Segmentation metrics labeled `*_fake_only` are the primary localization scores. `*_all_faces` includes real faces whose GT mask is zeros (both-empty counts as 1) and can look inflated.
8. No experimental number in this notebook is fabricated: unrun cells export `NOT RUN` / `N/A`.
